# 🔬 informed_flow — diagnostics: is a candidate worth trading?

[`informed_flow.ipynb`](./informed_flow.ipynb) optimizes and saves its best admissible trials as
candidates. This notebook takes **any set of those candidates** — plus what `config.yaml` ships
today, the **no-tuning control** — and puts them through every check that decides whether a
parameter set goes to production, **side by side**.

**How to use it.** In §0 set `CANDIDATES` (`"pick"`, trial numbers from the saved candidates, or
`"config"`) and `FOCUS` (the one candidate the per-name figures draw), then Run All. The checks
run for every candidate; the name-level figures (§2b, §2c, §5c) for `FOCUS` only.

| § | Question | Candidates |
|---|---|---|
| 2 Understand | Does the ranking carry information once each name's beta is removed — every year, in every liquidity group, and at the TOP of the ranking where a long-only book lives? Does the book hold what the ranking likes? | all / FOCUS |
| 3 Stress in-sample | Costs at the real order size, fill timing, year by year, parameter stability, a shuffled book, concentration | all |
| 4 Walk-forward | Does the **whole procedure** — re-read the signal rule on the data so far, hold the sleeves that pass, trade the fixed rules — survive unseen data, as one stitched record? | procedure |
| 5 Holdout | The 2025+ window — **used**, read under many study versions — and the shuffled-book test on it | all / FOCUS |
| 6 Verdict | One pass/fail table | all |
| 7 Promote | Write a passing candidate into `config.yaml`, stamped with where it came from | one |

**The gates** (thresholds in §0) answer three questions, in this order. The return that counts is
**alpha after beta** (`alpha_cagr = cagr − beta × buy-and-hold cagr`, the study's objective).

| | Gate | Pass when |
|---|---|---|
| **1 Is there skill?** | The ranking carries information, train | the composite's **beta-adjusted** rank-IC at `ic_horizon` bars has t ≥ `ic_min_t` (3 since v32), passes the calendar-year test (`year_test`: positive in ≥ `ic_min_positive_years` of the years, or — v33 — no full year with t < `ic_max_bad_year_t`) and is positive in ≥ `ic_min_positive_groups` of 3 dollar-volume groups |
| | Walk-forward | the stitched out-of-sample record of the **rule arm** (sleeves re-chosen inside every period from that period's own training data) has alpha > 0 with P(alpha > 0) ≥ `min_prob_alpha`, and at least half its full periods positive — tests the procedure, so the same for every candidate |
| | The signal is real, unseen data | the rule arm beats the 95th percentile of its own book run on **symbol-shuffled** rankings (within dollar-volume tenths), stitched over the whole walk-forward. The same test on the holdout alone is a **warning** since v32 |
| **2 Is it worth the risk?** | Walk-forward beats the market | stitched CAGR > buy-and-hold's **and** return ÷ volatility ≥ buy-and-hold's |
| | Risk limits | beta, max drawdown, volatility against **your own** limits — **warnings** |
| **3 Can it be run?** | Holds enough, train | completed holds per year ≥ `min_holds_per_year` (holds, not vectorbt lots — every re-trim splits a lot) |
| | Survives real costs | train alpha > 0 at the account's real order size and at `cost_multiple`× costs |
| | Survives real execution | train alpha > 0 when every leg fills `late_fill_bps` worse than the open (a hand-placed order in the first half hour), and with **whole shares** at `ACCOUNT` (since v32) |
| | Not a narrow peak | no parameter-stability `spike` |
| | Fill timing | train alpha under pessimistic fills ≥ `stress_min_alpha` — a **warning** |
| *warnings* | Holdout alone | alpha, its bootstrap, every year, the composite IC. The 2025+ window has been read under eight study versions, and with ~1.7 years of a 30-name book its interval is wide: a **PASS here would prove little and a FAIL little more** |

Every train number is scored from `optuna.score_from` and every holdout and walk-forward window
**warm** (`score_from`): the strategy runs on the whole history and only the window's bars
count, as they would live. **Do not tune against §4–§5.**

## 0 · Choose the candidates

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd
import yaml
from scipy import stats
from sqlalchemy import text

from trading_simple import backtest as bt
from trading_simple import features, plots, strategies
from trading_simple import optimize as opt
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

# --- what to diagnose ------------------------------------------------------------------------
CANDIDATES = ["config"]          # "config" = what config.yaml ships. Nothing is searched since
                                 # v28, so "pick" (or a saved trial number) is the same rules.
FOCUS = None               # the candidate the per-name figures draw (None = the first one)
STUDY = None               # None = config.yaml's optuna.study
# The last ADOPTED version, as overrides on today's config.yaml (params) and on GATES below (the
# signal rule). §4d re-runs it inside the SAME walk-forward and compares the two day by day.
# {} / {} = the run has no baseline (the first version of a line of work).
BASELINE_LABEL = "v33 rules, fixed Liquid500"   # v41: the whole night for the shipped config
BASELINE = {"dtc_window": 1, "min_events": 1, "max_positions": 30}
BASELINE_GATES = {}
# The universe each walk-forward period trades. "fixed" = data.universe (Liquid500: ranked on
# 2020-H1 AND required to have >= 1,000 sessions through 2024-12, a survival filter read at the end
# of train);
# "rolling" = Liquid500's recipe re-run at every period start on the 6 months before it (top 500
# equities by median dollar volume >= $50M, median price >= $5, >= 100 sessions): what a live book
# could actually have traded then. BASELINE_WF_UNIVERSE is the universe the §4d baseline arm trades.
WF_UNIVERSE = "rolling"
BASELINE_WF_UNIVERSE = "fixed"
ROLL = {"top_n": 500, "months": 6, "min_dollar_volume": 50e6, "min_price": 5.0, "min_days": 100}

# --- the account, the gates (§6), and effort ---------------------------------------------------
ACCOUNT = 30_000          # the REAL account in dollars. Portfolio mode is scale-free until a $1
                          # minimum commission exists; then the order size (ACCOUNT / slots) rules.
GATES = {
    # hard gates ---------------------------------------------------------------------------------
    "ic_horizon": 63,               # the signal rule: the composite's beta-adjusted rank-IC, here
    "ic_min_t": 3.0,                # ... its Newey-West t (v32: was 2.0 — ~Bonferroni for the five
                                    #     sleeves re-tested every six months; IDEAS.md)
    "year_test": "no_bad_year",     # ... "share": IC > 0 in >= ic_min_positive_years of the scored
                                    #     calendar years (v27-v32); "no_bad_year": no FULL calendar
                                    #     year with IC t < ic_max_bad_year_t (v33, pre-registered)
    "ic_min_positive_years": 0.8,   # ... share of scored train calendar years with IC > 0
    "ic_max_bad_year_t": -2.0,      # ... the worst full year's Newey-West t allowed ("no_bad_year")
    "ic_min_positive_groups": 2,    # ... of the 3 dollar-volume groups with IC > 0
    "min_prob_alpha": 0.90,         # bootstrap P(alpha after beta > 0) on the stitched walk-forward
    "wf_min_positive_share": 0.5,   # full walk-forward periods with out-of-sample alpha > 0
    "random_percentile": 95,        # beat this percentile of symbol-shuffled books on UNSEEN data
    "min_holds_per_year": 20,       # completed holds per scored train year (not vectorbt lots)
    "cost_multiple": 2.0,           # train alpha must stay > 0 at this multiple of the costs
    "late_fill_bps": 50.0,          # ... and with every leg this much worse than the open: the
                                    #     median distance of a first-half-hour fill from it (§5e)
    # warnings: your own risk limits, and the fill-timing stress ---------------------------------
    "max_beta": 1.5,                # holdout beta against equal-weight buy-and-hold
    "max_drawdown": -0.35,          # holdout max drawdown
    "max_vol_ratio": 2.0,           # holdout volatility / buy-and-hold's volatility
    "stress_min_alpha": -0.03,      # train alpha under pessimistic (buy-high / sell-low) fills
}
HOLDOUT_START = "2025-01-01"
WF_START, WF_STEP_MONTHS = "2022-01-01", 6   # first out-of-sample bar, and the refit cadence
WF_MIN_BARS = 60          # a shorter last period joins the stitched record but is not a "period"
WF_TRIALS = None          # trials per walk-forward study (None = optuna.n_trials, the same budget)
N_RANDOM_SIMS = 100       # symbol-shuffled books per candidate on TRAIN (information only)
N_OOS_SIMS = 500          # ... per shuffle kind on the holdout window (a gate, §5d)
N_WF_SIMS = 500           # ... and over the WHOLE walk-forward for the rule arm (a gate, §4)
N_BOOT, BOOT_BLOCK = 2000, 21   # stationary block bootstrap: resamples, mean block (bars)
COST_MULTIPLES = (1.0, 2.0, 3.0)

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("informed_flow")
d = strat.data
STUDY = STUDY or strat.optuna["study"]
SPACE = strat.optuna["space"]
SCORE_FROM = strat.optuna.get("score_from") or d["start"]   # the first TRAIN bar that counts
OBJECTIVES = opt.objectives_for(strat)
HURDLE = OBJECTIVES[0] if OBJECTIVES[0] in opt.HURDLES else "excess_cagr"
WF_TRIALS = WF_TRIALS or strat.optuna["n_trials"]
# Real execution (§3a, §4c, §5a): late fills pay late_fill_bps more slippage per leg; whole shares
# are sized at the REAL account (the engine's init_cash is only a reporting scale otherwise).
LATE_FILLS = settings.model_copy(update={"costs": settings.costs.model_copy(update={
    "slippage_bps": settings.costs.slippage_bps + GATES["late_fill_bps"]})})
AT_ACCOUNT = settings.model_copy(update={"backtest": settings.backtest.model_copy(update={
    "init_cash": ACCOUNT})})
LATE_COL = f"late fills (+{GATES['late_fill_bps']:g} bp/leg)"
WHOLE_COL = f"whole shares at ${ACCOUNT:,.0f}"


def whole_shares(params, strategy=None):
    """The strategy sized in whole shares at ACCOUNT / slots per order (pair it with AT_ACCOUNT)."""
    slots = {**strat.params, **params}["max_positions"]
    return replace(strategy or strat, whole_shares=True, order_notional=ACCOUNT / slots)

saved = yaml.safe_load(opt.study_path(strat, STUDY).with_suffix(".candidates.yaml").read_text())
by_trial = {row["trial"]: row for row in saved["front"]}
FLOOR = saved["pick_min_trades_per_year"]
if saved.get("score_from", SCORE_FROM) != SCORE_FROM:
    raise ValueError(f"candidates were scored from {saved['score_from']}, config says "
                     f"{SCORE_FROM}: re-run informed_flow.ipynb §7")


def resolve(name):
    """A candidate name -> (label, searched params, saved TRAIN metrics or {})."""
    if name == "config":
        return "config", {k: strat.params[k] for k in SPACE}, {}
    trial = saved["pick"] if name == "pick" else int(name)
    if trial not in by_trial:
        raise KeyError(f"trial {trial} is not among the saved candidates of {STUDY}; "
                       f"choose from {sorted(by_trial)}")
    row = by_trial[trial]
    label = f"pick #{trial}" if trial == saved["pick"] else f"#{trial}"
    return label, row["params"], {k: v for k, v in row.items() if k not in ("trial", "params")}


resolved = [resolve(c) for c in CANDIDATES]
cands = {label: params for label, params, _ in resolved}
TRIAL_OF = {label: (saved["pick"] if c == "pick" else None if c == "config" else int(c))
            for c, (label, _, _) in zip(CANDIDATES, resolved, strict=True)}
FOCUS = resolve(FOCUS)[0] if FOCUS is not None else next(iter(cands))
if FOCUS not in cands:
    raise KeyError(f"FOCUS {FOCUS!r} must be one of CANDIDATES {list(cands)}")
print(f"study {STUDY} · objectives {list(OBJECTIVES)} · scored from {SCORE_FROM} · candidates "
      f"saved {saved['saved']} · {len(by_trial)} saved candidates · pick #{saved['pick']} "
      f"· FOCUS = {FOCUS}")
if not saved.get("admissible", True):
    print(f"⚠ NO trial traded ≥ {FLOOR}/yr with alpha > 0: the pick fell back past the floor "
          "(§6 gates the completed holds)")
n_main = len(opt.load_study(strat, STUDY).trials)
if SPACE and n_main != WF_TRIALS:
    print(f"⚠ the main study holds {n_main} trials but each walk-forward refit gets {WF_TRIALS}: "
          "§4 then tests a different search than the one that made the pick")

if SPACE:
    params_table = pd.DataFrame(cands)
    params_table.insert(0, "config.yaml", [strat.params[k] for k in params_table.index])
    display(params_table.style.format(lambda v: f"{v:g}" if isinstance(v, float) else str(v))
            .apply(lambda r: ["" if c == "config.yaml" or r[c] == r["config.yaml"]
                              else "background: #fde3c0" for c in r.index], axis=1))
    print("shaded = differs from what config.yaml ships")
else:
    print("nothing is searched: every candidate is config.yaml's fixed rules")
if trained := {label: m for label, _, m in resolved if m}:
    print("\nsaved TRAIN metrics (informed_flow.ipynb §7):")
    display(pd.DataFrame(trained).style.format("{:.3f}"))

## 1 · Load the panels

Two loads, deliberately: the **train** panel exactly as the optimizer saw it, and a separate
**full-history** panel for every out-of-sample score. (Slicing train out of one long load would
leak holdout availability into training data: a feature column exists only where the query
window returned rows for that symbol.) `score()` is the one function every later section reads
its numbers from, and `bootstrap()` turns its daily returns into an interval.

**Why a bootstrap.** A holdout of ~1.7 years holds a few dozen completed positions, and a
walk-forward a handful of periods: every headline is a point estimate on a tiny sample. The
**stationary block bootstrap** (Politis & Romano 1994) resamples the book's and buy-and-hold's
daily returns *together*, in blocks of random length averaging `BOOT_BLOCK` bars — so the
volatility clustering and the book-to-market link survive — and recomputes alpha after beta on
each resample. `P(alpha > 0)` is the share of resamples where the candidate still earns it.

In [ ]:
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))


def load(end, symbols=None):
    """One panel from data.start to `end` (`symbols`, default the universe), features attached."""
    p = load_panel(engine, symbols or syms, d["bar"], d["start"], end,
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
    features.attach_features(p, engine, d["features"], settings=settings)
    return p


MEMBERS = {}


def members(te0):
    """The point-in-time top-N equities at `te0`, ranked only on the ROLL["months"] before it."""
    if str(te0) in MEMBERS:
        return MEMBERS[str(te0)]
    start = pd.Timestamp(te0) - pd.DateOffset(months=ROLL["months"])
    ranked = pd.read_sql(text("""
        SELECT p.symbol,
               percentile_cont(0.5) WITHIN GROUP (ORDER BY p.close * p.volume) AS dollar_volume,
               percentile_cont(0.5) WITHIN GROUP (ORDER BY p.close) AS price, count(*) AS days
        FROM prices_ohlcv p JOIN securities s ON s.symbol = p.symbol
        WHERE p.bar = '1d' AND p.ts >= :a AND p.ts < :b AND s.security_type = 'equity'
        GROUP BY p.symbol"""), engine, params={"a": start, "b": pd.Timestamp(te0)}
    ).set_index("symbol")
    pool = ranked[(ranked["days"] >= ROLL["min_days"])
                  & (ranked["dollar_volume"] >= ROLL["min_dollar_volume"])
                  & (ranked["price"] >= ROLL["min_price"])]
    MEMBERS[str(te0)] = sorted(pool.nlargest(ROLL["top_n"], "dollar_volume").index)
    return MEMBERS[str(te0)]


def pit_history(te0):
    """Every membership formed from optuna.score_from to `te0` (every ROLL["months"]): the panel
    a rule reading at `te0` may use, and on which DATE each name belonged to it.

    v36 read the rule on members(te0) alone — names chosen by liquidity at the END of the training
    window. That selects on the future: heavily shorted names that later surged are exactly the ones
    that became liquid, and the short-interest IC on 2020-07 -> 2023-12 fell from +0.057 (t 5.2,
    point-in-time) to +0.022 (t 1.7) — the rule then failed a sleeve that works.
    """
    origins = pd.date_range(SCORE_FROM, te0, freq=f"{ROLL['months']}MS")
    return {o: members(str(o.date())) for o in origins}


def member_mask(panel, history):
    """(bars x symbols) True where a name was a member ON that date (latest formation <= date)."""
    cols = panel.close.columns
    mask = pd.DataFrame(False, index=panel.close.index, columns=cols)
    first = min(history)
    for o, names in sorted(history.items()):
        rows = panel.close.index >= pd.Timestamp(o, tz="UTC") if o != first else slice(None)
        mask.loc[rows] = cols.isin(names)
    return mask


def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx],
                   features={k: v.loc[idx] for k, v in (p.features or {}).items()})


def score(params, p, start=None, strategy=None, cfg=None):
    """Every number the gates read, for one parameter set on one window.

    The engine's own metrics (`bt.evaluate`'s set, `alpha_cagr` and `beta` included — geometric,
    never an arithmetic mean times bars per year), run once through the same warm path, plus
    buy-and-hold's CAGR and volatility and both daily return series for the bootstrap. `start`
    scores WARM: signals on all of `p`, book, benchmark and metrics from `start` on.
    """
    strategy, cfg = strategy or strat, cfg or settings
    pf, w, fees, slip = bt._run_scored(strategy, p, params, cfg, start)
    out = bt.portfolio_stats(pf)
    out |= bt.active_stats(pf, w, fees=fees, slippage=slip, total_return=out["total_return"],
                           fill=strategy.fill)
    bench_ret, bench = bt.benchmark(w, fees=fees, slippage=slip, fill=strategy.fill)
    out["bench_cagr"] = bt._cagr(bench_ret, out["years"])
    out["bench_volatility"] = float(bt._bar_volatility(bench, w.close.index))
    out["ret_per_vol"] = out["cagr"] / out["volatility"] if out["volatility"] else np.nan
    out["bench_ret_per_vol"] = out["bench_cagr"] / out["bench_volatility"]
    out["returns"] = bt.equity_curve(pf)["value"].pct_change().fillna(0.0)
    out["bench_returns"] = bench
    out["pf"] = pf
    return out


def alpha_of(r, bench, years):
    """Geometric CAGR, buy-and-hold CAGR, beta and alpha after beta of aligned daily returns."""
    r, bench = np.atleast_2d(r), np.atleast_2d(bench)
    cagr = np.expm1(np.log1p(r).sum(axis=1) / years)
    bench_cagr = np.expm1(np.log1p(bench).sum(axis=1) / years)
    rc, bc = r - r.mean(axis=1, keepdims=True), bench - bench.mean(axis=1, keepdims=True)
    var = (bc * bc).sum(axis=1)
    beta = np.divide((rc * bc).sum(axis=1), var, out=np.zeros_like(var), where=var > 0)
    return cagr, bench_cagr, beta, cagr - beta * bench_cagr


def block_index(size, n=N_BOOT, block=BOOT_BLOCK, seed=7):
    """Stationary block-bootstrap resamples of positions 0..size-1, shape (n, size)."""
    rng = np.random.default_rng(seed)
    pos = np.arange(size)
    new = rng.random((n, size)) < 1.0 / block           # True where a new block starts
    new[:, 0] = True
    block_start = np.maximum.accumulate(np.where(new, pos, 0), axis=1)
    origin = np.take_along_axis(np.where(new, rng.integers(0, size, (n, size)), 0),
                                block_start, axis=1)
    return (origin + pos - block_start) % size          # each block walks on from its origin


def bootstrap(r, bench, years, n=N_BOOT, block=BOOT_BLOCK, seed=7):
    """Stationary block bootstrap of alpha after beta and excess: P(alpha > 0) and a 90% CI."""
    r, bench = np.asarray(r, dtype=float), np.asarray(bench, dtype=float)
    idx = block_index(len(r), n, block, seed)
    cagr, bench_cagr, _, alpha = alpha_of(r[idx], bench[idx], years)
    excess = cagr - bench_cagr
    return {"P(alpha > 0)": float((alpha > 0).mean()),
            "alpha p5": float(np.percentile(alpha, 5)),
            "alpha p95": float(np.percentile(alpha, 95)),
            "P(excess > 0)": float((excess > 0).mean())}


def paired(r, base, bench, years, n=N_BOOT, block=BOOT_BLOCK, seed=11, base_bench=None):
    """Alpha of `r` minus alpha of `base` on the SAME days, both bootstrapped on the SAME blocks.

    Two separate intervals overlap even when one book beats the other on most days; resampling
    the pair together keeps what they share (the market, the same names) out of the comparison.
    """
    r, base, bench = (np.asarray(x, dtype=float) for x in (r, base, bench))
    bb = bench if base_bench is None else np.asarray(base_bench, dtype=float)   # its own market
    point = float(alpha_of(r, bench, years)[3][0] - alpha_of(base, bb, years)[3][0])
    idx = block_index(len(r), n, block, seed)
    diff = alpha_of(r[idx], bench[idx], years)[3] - alpha_of(base[idx], bb[idx], years)[3]
    return {"Δ alpha": point, "P(Δ > 0)": float((diff > 0).mean()),
            "Δ p5": float(np.percentile(diff, 5)), "Δ p95": float(np.percentile(diff, 95))}


PCT = "{:+.1%}"
panel = load(d.get("end"))          # TRAIN — what the study was fitted on
full = load(None)                   # the whole history — every warm out-of-sample score
LAST = str(full.close.index[-1].date())
holdout = slice_panel(full, HOLDOUT_START, LAST)    # charts only — never backtest a slice
try:        # what you could actually buy instead: an S&P 500 index fund, total return
    SPY = load_panel(engine, ["SPY"], d["bar"], d["start"], None,
                     session=d.get("session", "regular"),
                     exchanges=settings.exchanges).close["SPY"].dropna()
except Exception:                                       # not in this database: the rows stay empty
    SPY = None


def window_cagr(prices, start, end=None):
    """Geometric CAGR of a price series over [start, end] (NaN when the series is missing)."""
    if prices is None:
        return np.nan
    s = prices.loc[pd.Timestamp(start, tz="UTC"):pd.Timestamp(end or LAST, tz="UTC")]
    if len(s) < 2:
        return np.nan
    return float((s.iloc[-1] / s.iloc[0]) ** (1.0 / bt._years(s.index)) - 1.0)


print(f"{d['universe']}: {len(syms)} symbols · train {panel.close.index[0].date()} → "
      f"{panel.close.index[-1].date()}, scored from {SCORE_FROM} · holdout {HOLDOUT_START} → "
      f"{LAST} ({len(holdout.close)} bars)")

## 2 · Understand the candidates

### 2a · Does the ranking pick winners? (train)

A backtest mixes three things: whether the ranking picks winners, how the book is built (slots,
exits, costs), and simply being in a rising market. These tables measure **only the first**,
without trading anything.

**Table 1 — the plain rank check**, one row per ingredient (sleeve) plus the combined ranking,
at 1, 3 and 6 months ahead:

- **IC** — rank correlation between the score and the forward return, across every ranked name.
  Real signals here are small: +0.03 is already respectable.
- **top k** — the demeaned forward return of the **k highest-ranked names**, k = the book's
  `max_positions`: the population a long-only book can actually own.
- **t** — Newey–West t-statistics (Bartlett weights, lag = twice the horizon): consecutive forward
  windows overlap by all but one bar. Below 2: can't tell it apart from luck — and read 2 as a
  weak bar on samples this short.

**Table 2 — the same check made fair**, at `GATES["ic_horizon"]` bars. The plain IC has a blind
spot: a rank correlation **does not change** when the market's average is subtracted, so
"demeaned" removes nothing from it, and a ranking that merely leans toward high-beta names looks
skilled in a rising market (a pure-beta ranking scores ≈ +0.02 when the market rises 20%/yr).
So the forward return here is **beta-adjusted** — each name's trailing 252-bar beta times the
market's forward return is removed — and the IC is also read **per calendar year** and **inside
three dollar-volume groups** (a sleeve that only works across groups is a "heavily traded names"
tilt, not information). **Table 3** splits the ranking into fifths: a long-only book can only use
information that sits at the **top**.

The composite's row of Table 2 is the **signal rule**, a hard gate (§6). It is also the rule that
decided, on these train years and before v27 existed, which sleeves ship at all (`config.yaml`).

In [ ]:
HORIZONS = (21, 63, 126)          # ~1, 3 and 6 months ahead
HORIZON_NAMES = {21: "1 month", 63: "3 months", 126: "6 months"}
mod = strat.signals.__globals__
compose = mod["_compose"]


def nw_t(x, lag):
    """Newey-West (Bartlett) t-statistic of a series' mean — overlapping windows autocorrelate."""
    x = pd.Series(x).dropna().to_numpy(dtype=float)
    n = len(x)
    if n < 3:
        return np.nan
    e = x - x.mean()
    var = e @ e / n
    for j in range(1, min(lag, n - 1) + 1):
        var += 2.0 * (1.0 - j / (lag + 1.0)) * (e[j:] @ e[:-j]) / n
    return float(x.mean() / np.sqrt(var / n)) if var > 0 else np.nan


def signal_quality(score_frame, close, k):
    """Rank IC and the top-k names' demeaned forward return, each with a Newey-West t (lag 2h)."""
    rows, top = [], score_frame.rank(axis=1, ascending=False, method="first") <= k
    for h in HORIZONS:
        fwd = close.shift(-h) / close - 1.0
        rel = fwd.sub(fwd.mean(axis=1), axis=0)              # the market, removed
        ic = score_frame.corrwith(rel, axis=1, method="spearman").dropna()
        top_k = rel.where(top & score_frame.notna()).mean(axis=1).dropna()
        rows.append({"horizon": h, "IC": float(ic.mean()), "t": nw_t(ic, 2 * h),
                     "top k": float(top_k.mean()), "t top": nw_t(top_k, 2 * h)})
    return pd.DataFrame(rows).set_index("horizon")


def sleeve_table(p, params, start=None):
    """IC, top-k return and their t per sleeve and horizon; `start` scores warm."""
    full_params = {**strat.params, **params}
    parts = compose(p, **full_params)
    close = p.close if start is None else p.close.loc[pd.Timestamp(start, tz="UTC"):]
    rows = {}
    for key in parts["present"] + ["COMPOSITE"]:
        frame = parts["score_pct"] if key == "COMPOSITE" else parts["ranks"][key]
        q = signal_quality(frame.loc[close.index], close, full_params["max_positions"])
        rows[key] = {(HORIZON_NAMES[h], col): q.loc[h, col] for h in HORIZONS for col in q}
    return pd.DataFrame(rows).T


def show_ic(tables, window):
    """One table per candidate: IC, top-k return and t at every horizon."""
    for label, table in tables.items():
        k = {**strat.params, **cands[label]}["max_positions"]
        print(f"{window} — {label} (top k = {k})")
        top = [c for c in table.columns if c[1] == "top k"]
        t_cols = [c for c in table.columns if c[1] in ("t", "t top")]   # "top k" is a return
        display(table.style.format("{:+.3f}").format(PCT, subset=top)
                .format("{:+.2f}", subset=t_cols))


def trailing_beta(close, window=252):
    """Each name's beta to the equal-weight panel over the trailing `window` bars (known at t)."""
    r = close.pct_change(fill_method=None)
    m = r.mean(axis=1)
    return r.rolling(window, min_periods=window // 2).cov(m).div(
        m.rolling(window, min_periods=window // 2).var(), axis=0)


def residual_forward(close, h):
    """Forward `h`-bar return minus trailing beta x the market's forward return.

    Built on the WHOLE panel so the beta is warm on a window's first bar; the beta at bar t
    reads returns up to t only, so nothing here looks ahead of the score it is compared with.
    """
    fwd = close.shift(-h) / close - 1.0
    return fwd - trailing_beta(close).mul(fwd.mean(axis=1), axis=0)


def rank_ic(score_frame, rel):
    """Daily Spearman IC as Pearson on jointly masked row ranks — pandas' spearman, ~10x faster."""
    m = score_frame.notna() & rel.notna()
    return score_frame.where(m).rank(axis=1).corrwith(rel.where(m).rank(axis=1), axis=1).dropna()


def fair_check(p, params, start, h=None, mask=None):
    """The rank check made fair: beta-adjusted, per calendar year, per dollar-volume group.

    Returns:
        (summary, by_year, fifths) — one row per sleeve plus COMPOSITE. `fifths` is the average
        beta-adjusted forward return of each fifth of the ranking against the average name, and
        of the daily top `max_positions`.
    """
    h = h or GATES["ic_horizon"]
    fp = {**strat.params, **params}
    parts = compose(p, **fp)
    idx = p.close.index[p.close.index >= pd.Timestamp(start, tz="UTC")]
    rel = residual_forward(p.close, h).loc[idx]
    if mask is not None:                     # only the names that were members on each date
        rel = rel.where(mask.loc[idx])
    rel = rel.sub(rel.mean(axis=1), axis=0)
    dollar_volume = (p.close * p.volume).rolling(63, min_periods=21).median()
    group = np.ceil(dollar_volume.rank(axis=1, pct=True) * 3).clip(1, 3).loc[idx]
    summary, by_year, fifths = {}, {}, {}
    for key in parts["present"] + ["COMPOSITE"]:
        frame = (parts["score_pct"] if key == "COMPOSITE" else parts["ranks"][key]).loc[idx]
        if mask is not None:
            frame = frame.where(mask.loc[idx])
        ic = rank_ic(frame, rel)
        by_year[key] = ic.groupby(ic.index.year).mean()
        # a FULL calendar year: >= 200 daily ICs (2020 is scored from July; the last year is short)
        year_t = ic.groupby(ic.index.year).agg(
            lambda x: nw_t(x, 2 * h) if len(x) >= 200 else np.nan)
        in_group = [rank_ic(frame.where(group == g), rel.where(group == g)).mean()
                    for g in (1, 2, 3)]
        summary[key] = {"IC": ic.mean(), "t": nw_t(ic, 2 * h),
                        "years > 0": float((by_year[key] > 0).mean()),
                        "worst full-year t": float(year_t.min()),
                        "groups > 0": int(sum(x > 0 for x in in_group)),
                        **{f"group {g}": x for g, x in zip((1, 2, 3), in_group, strict=True)}}
        pct = frame.rank(axis=1, pct=True)
        row = {f"fifth {q}": rel.where((pct > (q - 1) / 5) & (pct <= q / 5)).mean(axis=1).mean()
               for q in range(1, 6)}
        top = rel.where((frame.rank(axis=1, ascending=False, method="first")
                         <= fp["max_positions"]) & frame.notna()).mean(axis=1)
        fifths[key] = row | {f"top {fp['max_positions']}": top.mean(), "t top": nw_t(top, 2 * h)}
    return pd.DataFrame(summary).T, pd.DataFrame(by_year).T, pd.DataFrame(fifths).T


def show_fair(tables, window):
    """Tables 2 and 3 per candidate: the fair IC with its year/group split, then the fifths."""
    h = GATES["ic_horizon"]
    for label, (summary, by_year, fifths) in tables.items():
        print(f"{window} — {label}: beta-adjusted rank-IC, {h} bars ahead "
              "(group 1 = least traded third, 3 = most traded)")
        display(pd.concat([summary, by_year.add_prefix("IC ")], axis=1).style.format("{:+.3f}")
                .format("{:+.2f}", subset=["t", "worst full-year t"])
                .format("{:.0%}", subset=["years > 0"])
                .format("{:.0f}", subset=["groups > 0"]))
        print(f"{window} — {label}: beta-adjusted {h}-bar forward return by fifth of the ranking "
              "(fifth 5 = best-ranked), against the average name")
        display(fifths.style.format(PCT).format("{:+.2f}", subset=["t top"]))


def year_test(summary):
    """The rule's calendar-year test, as `GATES["year_test"]` pre-registers it.

    "share" (v27-v32) wants IC > 0 in a share of the scored calendar years — a test that gets
    stricter the fewer years there are, and fails a year at -0.0004 like one at -0.05 (v30).
    "no_bad_year" (v33) only refuses a sleeve that had a FULL calendar year significantly wrong;
    with no full year yet there is nothing to refuse on.
    """
    if GATES["year_test"] == "share":
        return summary["years > 0"] >= GATES["ic_min_positive_years"]
    return not summary["worst full-year t"] < GATES["ic_max_bad_year_t"]    # NaN passes


def signal_rule(summary, years_test=True):
    """The pre-registered signal rule on one row of a fair-check summary -> (passes, text).

    `years_test=False` drops the calendar-year test — only for the §4 control arm that shows what
    that test costs on short histories; the gates always read the rule as pre-registered.
    """
    ok = (summary["t"] >= GATES["ic_min_t"] and summary["IC"] > 0
          and (not years_test or year_test(summary))
          and summary["groups > 0"] >= GATES["ic_min_positive_groups"])
    return bool(ok), (f"IC {summary['IC']:+.3f} · t {summary['t']:+.1f} · "
                      f"{summary['years > 0']:.0%} of years (worst full-year t "
                      f"{summary['worst full-year t']:+.1f}) · "
                      f"{summary['groups > 0']:.0f}/3 groups")


show_ic({label: sleeve_table(panel, p, start=SCORE_FROM) for label, p in cands.items()},
        "TRAIN")
fair_train = {label: fair_check(panel, p, SCORE_FROM) for label, p in cands.items()}
show_fair(fair_train, "TRAIN")
# The rule sleeve by sleeve, at EQUAL weights — the reading that decides which sleeves config.yaml
# ships. A sleeve at weight 0 is absent from the tables above, so every weight is switched on here.
ALL_ON = {k: 1.0 for k in ("w_insider", "w_rating", "w_ptrev", "w_dtc", "w_recency")}
every_sleeve = fair_check(panel, {**cands[FOCUS], **ALL_ON}, SCORE_FROM)[0].drop("COMPOSITE")
print("the signal rule, sleeve by sleeve (all five switched on; train only):")
WEIGHT_OF = {"insider": "w_insider", "ratings": "w_rating", "price_target": "w_ptrev",
             "short_interest": "w_dtc", "recency": "w_recency"}
display(pd.DataFrame({k: {"passes": signal_rule(row)[0], "reading": signal_rule(row)[1],
                          "ships": strat.params[WEIGHT_OF[k]] > 0}
                      for k, row in every_sleeve.iterrows()}).T)

### 2b · Why is a name in or out of the book? (FOCUS)

A name is bought when its **cross-sectional percentile rank** crosses `q_in` and a slot is free,
and sold when the rank falls through `q_out`. `strategy.explain()` returns the audit trail and
shares its whole computation with `signals()`, so this picture cannot drift from what traded.
Three things it makes visible, each of which looks like a bug until you see it:

1. **A name can cross with no event of its own** — the rank is relative, so it rises when its
   peers fall. The `own`/`peer` split attributes every rank change exactly.
2. **A name's own value moves with no new event** — an old insider buy rolling off the back of
   its trailing window moves the score by itself.
3. **A missing signal is a neutral 0.5, not a zero** — hatched bars carry no information.

Hollow circle = qualified, ✕ = qualified but no free slot, ▲ = the actual fill (features lag a
bar and fills are next-open, so the fill trails the crossing); shaded spans are real positions.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple.plots import DIVERGING, SERIES

explain = mod["explain"]
PARAMS = {**strat.params, **cands[FOCUS]}
PAD = 60                       # bars either side of the crossing we zoom on

res_focus = bt.run_strategy(strat, panel, params=cands[FOCUS], settings=settings,
                            score_from=SCORE_FROM)
trades = bt.trade_records(res_focus)
# Explain the name the strategy traded most — that is where there is something to see.
SYMBOL = (trades["symbol"].value_counts().idxmax() if len(trades)
          else panel.features["insider"].abs().sum().idxmax())

exp = explain(panel, SYMBOL, **PARAMS)
mine = trades[trades["symbol"] == SYMBOL]

# Zoom on the first entry; fall back to the strongest rank jump if it never traded.
focus = (pd.Timestamp(mine["entry_ts"].iloc[0]) if len(mine)
         else exp["d_score"].abs().idxmax())
i_focus = exp.index.get_indexer([focus], method="nearest")[0]
start, stop = max(i_focus - PAD, 0), i_focus + PAD
# The slow trend SMA has no value until `trend_slow` bars exist: reach past its warm-up so the
# fast/slow crossover in row ④ is actually visible.
stop = max(stop, PARAMS["trend_slow"] + PAD)
win = exp.iloc[start: min(stop, len(exp))]

print(f"{FOCUS} · {SYMBOL}: {len(mine)} trades, zooming {win.index[0].date()} → "
      f"{win.index[-1].date()} ({len(win)} bars)")
print(f"sleeves in the composite: {[c[5:] for c in exp.columns if c.startswith('rank_')]}")
if dropped := set(panel.features) - {c[5:] for c in exp.columns if c.startswith("rank_")}:
    print(f"⚠ dropped (zero weight, or below the coverage floor): {sorted(dropped)}")

In [ ]:
SLEEVES = [c[5:] for c in exp.columns if c.startswith("rank_")]
q_in, q_out = PARAMS["q_in"], PARAMS["q_out"]

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, row_heights=[0.30, 0.22, 0.24, 0.24],
    vertical_spacing=0.05,
    specs=[[{}], [{}], [{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=("① Composite rank vs the q_in / q_out bands",
                    "② Why: each sleeve's weighted contribution (stacked = composite)",
                    "③ The events that moved it, as the strategy saw them",
                    "④ Price regime: close, fast & slow SMAs (the trend sleeve) and RSI"))
fig.update_annotations(font_size=13)

# --- ① the rank the bands are compared against ----------------------------------------------
fig.add_trace(go.Scatter(x=win.index, y=win["score_pct"], name="composite rank",
                         line={"color": SERIES[0], "width": 2},
                         customdata=np.c_[win["own"], win["peer"], win["driver"]],
                         hovertemplate=("rank %{y:.2f}<br>own %{customdata[0]:+.3f}"
                                        " · peer %{customdata[1]:+.3f}"
                                        " → %{customdata[2]}-driven<extra></extra>")),
              row=1, col=1)
cross = win.index[win["qualified"]]
fig.add_trace(go.Scatter(
    x=cross, y=win["score_pct"].reindex(cross), mode="markers", name="qualified",
    marker={"symbol": "circle-open", "size": 13, "color": SERIES[3],
            "line": {"width": 2, "color": SERIES[3]}}), row=1, col=1)
# A "-thin" symbol has no fill: plotly strokes it from marker.line.color.
if (turned_away := win.index[win["slot_blocked"]]).size:
    fig.add_trace(go.Scatter(
        x=turned_away, y=win["score_pct"].reindex(turned_away), mode="markers",
        name="NO SLOT (book full)",
        marker={"symbol": "x-thin", "size": 10, "color": "#c0392b",
                "line": {"width": 2.5, "color": "#c0392b"}}), row=1, col=1)
w0, w1 = win.index[0], win.index[-1]
# vectorbt records one row per LOT: dedupe to the bars the book actually opened on.
fills = pd.DatetimeIndex(sorted(set(pd.to_datetime(mine["entry_ts"])))).intersection(win.index)
fig.add_trace(go.Scatter(
    x=fills, y=win["score_pct"].reindex(fills), mode="markers", name="actual fill",
    marker={"symbol": "triangle-up", "size": 15, "color": SERIES[4],
            "line": {"width": 1.5, "color": "#ffffff"}}), row=1, col=1)
if (blocked := win.index[~win["eligible"]]).size:
    fig.add_trace(go.Scatter(
        x=blocked, y=np.zeros(len(blocked)), mode="markers", name="ineligible (min_events)",
        marker={"symbol": "line-ns-open", "size": 8, "color": "#898781",
                "line": {"width": 1.5, "color": "#898781"}}), row=1, col=1)

# --- ② attribution; hatched = neutral-filled (NO DATA) --------------------------------------
for i, k in enumerate(SLEEVES):
    gap = win[f"filled_{k}"]
    fig.add_trace(go.Bar(
        x=win.index, y=win[f"contrib_{k}"], name=k, marker_color=SERIES[i],
        marker_pattern={"shape": ["/" if g else "" for g in gap],
                        "fgcolor": "#ffffff", "solidity": 0.35},
        customdata=np.c_[win[f"rank_{k}"], gap],
        hovertemplate=(f"<b>{k}</b><br>contribution %{{y:.3f}}<br>rank %{{customdata[0]:.2f}}"
                       "<br>no data (neutral 0.5): %{customdata[1]}<extra></extra>")),
        row=2, col=1)
fig.add_trace(go.Scatter(x=win.index, y=win["composite"], name="composite",
                         line={"color": "#0b0b0b", "width": 1.5}), row=2, col=1)

# --- ③ the raw point-in-time events ---------------------------------------------------------
if "insider_usd" in win:
    v = win["insider_usd"].fillna(0)
    fig.add_trace(go.Bar(x=win.index, y=v, name="insider net $",
                         marker_color=[DIVERGING[2][1] if x > 0 else DIVERGING[0][1] for x in v],
                         hovertemplate="insider $%{y:,.0f}<extra></extra>"), row=3, col=1)
for col, label, shape in [("rating_change", "rating Δ", "triangle-up"),
                          ("ptrev", "price-target rev", "diamond")]:
    if col not in win:
        continue
    ev = win[col].fillna(0)
    ev = ev[ev != 0]
    fig.add_trace(go.Scatter(
        x=ev.index, y=np.zeros(len(ev)), mode="markers", name=label,
        marker={"symbol": [shape if x > 0 else shape.replace("-up", "-down") for x in ev],
                "size": 10, "color": [SERIES[3] if x > 0 else SERIES[5] for x in ev]},
        customdata=ev.to_numpy(),
        hovertemplate=f"{label} %{{customdata:+.3f}}<extra></extra>"), row=3, col=1)
if "days_to_cover" in win:
    fig.add_trace(go.Scatter(x=win.index, y=win["days_to_cover"], name="days-to-cover",
                             line={"color": SERIES[6], "width": 1.5, "shape": "hv"}),
                  row=3, col=1, secondary_y=True)

# --- ④ price regime: close, the two SMAs of the trend sleeve, RSI ----------------------------
_rsi = mod["_rsi"]
tf, ts, rp = PARAMS["trend_fast"], PARAMS["trend_slow"], PARAMS["rsi_period"]
close_w = panel.close[SYMBOL].reindex(win.index)
fast_w = bt.sma(panel.close, tf)[SYMBOL].reindex(win.index)
slow_w = bt.sma(panel.close, ts)[SYMBOL].reindex(win.index)
rsi_w = _rsi(panel.close, rp)[SYMBOL].reindex(win.index)
fig.add_trace(go.Scatter(x=win.index, y=close_w, name="close (reference)",
                         line={"color": "#9b988f", "width": 1.3},
                         hovertemplate="close $%{y:,.2f}<extra></extra>"), row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=fast_w, name=f"fast SMA {tf}",
                         line={"color": SERIES[0], "width": 2.2},
                         hovertemplate=f"fast SMA {tf} $%{{y:,.2f}}<extra></extra>"),
              row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=slow_w, name=f"slow SMA {ts}",
                         line={"color": SERIES[7], "width": 2.2, "dash": "dash"},
                         hovertemplate=f"slow SMA {ts} $%{{y:,.2f}}<extra></extra>"),
              row=4, col=1)
fig.add_trace(go.Scatter(x=win.index, y=rsi_w, name=f"RSI {rp}",
                         line={"color": SERIES[6], "width": 1.5},
                         hovertemplate="RSI %{y:.0f}<extra></extra>"),
              row=4, col=1, secondary_y=True)
# Guides and shapes go on AFTER the traces: add_hline/add_hrect/add_vrect silently drop a shape
# aimed at a subplot that has no trace yet.
for lvl, tag, pos in [(70, "RSI 70 · overbought", "top right"),
                      (30, "RSI 30 · oversold", "bottom right")]:
    fig.add_hline(y=lvl, line={"color": "#b0ada5", "width": 1, "dash": "dot"},
                  row=4, col=1, secondary_y=True,
                  annotation_text=tag, annotation_position=pos,
                  annotation_font_size=10, annotation_font_color="#898781")
fig.add_hrect(y0=q_in, y1=1.0, fillcolor=SERIES[1], opacity=0.10, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"IN (rank ≥ q_in = {q_in})", annotation_position="top left",
              annotation_font_size=11, annotation_font_color="#898781")
fig.add_hrect(y0=q_out, y1=q_in, fillcolor=SERIES[2], opacity=0.09, line_width=0,
              layer="below", row=1, col=1,
              annotation_text=f"HOLD band (q_out = {q_out} – {q_in})",
              annotation_position="bottom left",
              annotation_font_size=11, annotation_font_color="#898781")
# REAL positions from the portfolio: union the FIFO lots into the spans the book was long.
lots = mine[["entry_ts", "exit_ts"]].apply(pd.to_datetime).sort_values("entry_ts")
held: list[list[pd.Timestamp]] = []
for entry, exit_ in lots.itertuples(index=False):
    if held and entry <= held[-1][1]:
        held[-1][1] = max(held[-1][1], exit_)
    else:
        held.append([entry, exit_])
for entry, exit_ in held:
    if exit_ >= w0 and entry <= w1:
        fig.add_vrect(x0=max(entry, w0), x1=min(exit_, w1),
                      fillcolor=SERIES[3], opacity=0.08, line_width=0, layer="below")

fig.update_layout(
    title={"text": (f"<b>{SYMBOL}</b> — why it is in or out of the book ({FOCUS})<br>"
                    "<sup>○ qualified · ✕ no slot · ▲ actual fill · shaded span = position held"
                    " · hatched bar = NO DATA, neutral 0.5</sup>"),
           "font": {"size": 15, "color": "#0b0b0b"},
           "x": 0, "xref": "paper", "xanchor": "left",
           "y": 0.99, "yref": "container", "yanchor": "top"},
    template="plotly_white", height=1200, barmode="relative", bargap=0.05,
    font={"family": 'system-ui, -apple-system, "Segoe UI", sans-serif', "size": 12},
    hovermode="x unified",
    legend={"orientation": "h", "x": 0.5, "xanchor": "center",
            "y": 1.06, "yanchor": "bottom", "font": {"size": 11}},
    margin={"l": 70, "r": 70, "t": 270, "b": 50})
fig.update_xaxes(range=[w0, w1])       # pin the zoom: a shape must not drag the autorange
fig.update_yaxes(title_text="rank (0–1)", range=[-0.02, 1.02], row=1, col=1)
fig.update_yaxes(title_text="contribution", row=2, col=1)
fig.update_yaxes(title_text="insider $ / events", row=3, col=1)
fig.update_yaxes(title_text="days-to-cover", showgrid=False, row=3, col=1, secondary_y=True)
fig.update_yaxes(title_text="price ($)", row=4, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], showgrid=False, row=4, col=1,
                 secondary_y=True)
fig.show()

# Every bar this name crossed a band, with the cause attributed. driver == "peer" and
# own_event == False is the counter-intuitive case: the cross-section moved under the name.
zone_prev = exp["zone"].shift(1)
crossings = exp[(exp["zone"] != zone_prev) & zone_prev.notna()].copy()
crossings["move"] = zone_prev[crossings.index] + " → " + crossings["zone"]
cols = ["move", "score_pct", "d_score", "own", "peer", "driver", "own_event", "eligible"]
display(crossings[cols + [f"d_contrib_{k}" for k in SLEEVES]].round(3).tail(12))
peer_driven = crossings[(crossings["driver"] == "peer") & ~crossings["own_event"]]
print(f"{len(crossings)} band crossings for {SYMBOL}; {len(peer_driven)} were pure PEER DRIFT "
      "— the name crossed with no news of its own.")

### 2c · The whole book at once — who is held, and when (FOCUS)

Every name the book held at some point in the scored window, and its
daily state: **held** (a real position), **qualified — no slot** (ranked above
`q_in` but the book was full: the capital constraint, and the answer to "its rank is 0.95, why
do I not own it?"), **band** (inside the `q_out`–`q_in` hysteresis zone), **out**, and
**ineligible** (too few events to be ranked at all). Read it for **churn** (names flickering in
and out pay costs for nothing), **breadth** (the same handful always held means the
cross-sectional thesis is not doing the work) and **pressure** (a wall of no-slot days means the
ranking has more conviction than the book has room for).

In [ ]:
q_in, q_out = PARAMS["q_in"], PARAMS["q_out"]
parts = compose(panel, **PARAMS)
selected = mod["_select"](parts, q_in=q_in, q_out=q_out, max_hold=PARAMS["max_hold"],
                          max_positions=PARAMS["max_positions"],
                          displace_margin=PARAMS["displace_margin"])
score_pct, eligible = parts["score_pct"], parts["eligible"]

# 0 ineligible · 1 out · 2 band · 3 qualified but NO SLOT · 4 held (a NaN rank falls to 1, then 0)
state = np.where(score_pct >= q_in, 3, np.where(score_pct >= q_out, 2, 1))
state = np.where(eligible.to_numpy(), state, 0)
state = pd.DataFrame(state, index=score_pct.index, columns=score_pct.columns)
state = state.mask(selected["slot_blocked"], 3)
state = state.loc[state.index >= pd.Timestamp(SCORE_FROM, tz="UTC")]   # the scored window
for _, t in trades.iterrows():   # truth beats reconstruction
    state.loc[pd.Timestamp(t["entry_ts"]):pd.Timestamp(t["exit_ts"]), t["symbol"]] = 4

# Hundreds of names do not fit on a screen: draw every name the book ever held, most-held first.
held_days = state.eq(4).sum()
order = held_days[held_days > 0].sort_values(ascending=False).index
z = state[order].T
LABELS = ["ineligible", "out", "band", "qualified — NO SLOT", "held"]
COLORS = ["#e8e6df", "#f4f3ef", "#fde3c0", "#e8a49c", SERIES[3]]
bands = [[i / 5, COLORS[j]] for j in range(5) for i in (j, j + 1)]
fig = go.Figure(go.Heatmap(
    z=z.to_numpy(), x=z.columns, y=z.index, colorscale=bands, zmin=0, zmax=5,
    colorbar={"tickvals": [0.5, 1.5, 2.5, 3.5, 4.5], "ticktext": LABELS, "len": 0.6},
    hovertemplate="%{y} · %{x|%Y-%m-%d}<extra></extra>"))
fig.update_layout(
    title=(f"Book state by name ({FOCUS}) — {len(order)} of {state.shape[1]} names held · "
           f"q_in={q_in}, "
           f"q_out={q_out} · {PARAMS['max_positions']} slots · held spans are real trades"),
    template="plotly_white", height=max(460, 11 * len(order)),
    margin={"l": 70, "r": 20, "t": 70, "b": 40})
fig.update_yaxes(tickfont={"size": 8})
fig.show()
print(f"held: {int(state.eq(4).sum().sum()):,} symbol-days across "
      f"{int(state.eq(4).any().sum())} names · qualified but NO SLOT on "
      f"{selected['slot_blocked'].loc[state.index].sum(axis=1).mean():.1f} names an average day · "
      f"{parts['n_ranked'].loc[state.index].mean():.0f} of {len(score_pct.columns)} names "
      "ranked per day")

### 2d · Does the book hold what the ranking likes?

§2a's top-k column describes the k best-ranked names **each day**. The book is something else: it
buys a top name and keeps it until its rank falls through `q_out` (or the `max_hold` clock runs
out), so what it owns is a mix of fresh top names and names sliding down the ranking. This table
puts three groups on the same beta-adjusted forward return: the **daily top k**, what the book
actually **held**, and the names that **qualified but found no slot**. If *held* earns much less
than *top k*, the exit band is too loose — the book is carrying names the ranking stopped liking.
(The v26 pick sold at a median rank of 0.49, while only the top fifth of the ranking pays.)

In [ ]:
def book_vs_ranking(p, params, start, h=None):
    """Mean rank and beta-adjusted forward return: daily top-k vs HELD vs qualified-without-a-slot.

    `_select` carries state from bar to bar, so it always runs on the whole panel and is cut to
    the window afterwards; the held mask is on the signal clock, like the ranks it is compared to.
    """
    h = h or GATES["ic_horizon"]
    fp = {**strat.params, **params}
    parts = compose(p, **fp)
    sel = mod["_select"](parts, q_in=fp["q_in"], q_out=fp["q_out"], max_hold=fp["max_hold"],
                         max_positions=fp["max_positions"], displace_margin=fp["displace_margin"])
    idx = p.close.index[p.close.index >= pd.Timestamp(start, tz="UTC")]
    ranks = parts["score_pct"].loc[idx]
    rel = residual_forward(p.close, h).loc[idx]
    rel = rel.sub(rel.mean(axis=1), axis=0)
    k = fp["max_positions"]
    groups = {f"top {k} today": ranks.rank(axis=1, ascending=False, method="first") <= k,
              "held": sel["held"].loc[idx], "qualified, no slot": sel["slot_blocked"].loc[idx]}
    out = {}
    for name, mask in groups.items():
        ret = rel.where(mask).mean(axis=1)
        out[name] = {"names / day": mask.sum(axis=1).mean(),
                     "mean rank": float(np.nanmean(ranks.where(mask).to_numpy())),
                     f"forward {h} bars": ret.mean(), "t": nw_t(ret, 2 * h)}
    return pd.DataFrame(out).T


def show_book(tables, window):
    """One held-vs-ranking table per candidate."""
    for label, table in tables.items():
        print(f"{window} — {label}")
        display(table.style.format({"names / day": "{:.0f}", "mean rank": "{:.2f}", "t": "{:+.2f}",
                                    f"forward {GATES['ic_horizon']} bars": PCT}))


show_book({label: book_vs_ranking(panel, p, SCORE_FROM) for label, p in cands.items()}, "TRAIN")

## 3 · Stress the candidates in-sample (train)

Everything in this section reads the training window only — the one the study was fitted on,
scored from `optuna.score_from` exactly as the study scored it — so it can show a candidate is
fragile, never that it is good. First the baseline comparison: each candidate against
equal-weight buy-and-hold of the same names, with the numbers the gates read. **alpha** is the
engine's `alpha_cagr`, `cagr − beta × buy-and-hold cagr`: the return left after the market
exposure the book actually ran. A book with beta 1.3 that beats buy-and-hold by less than 30% of
buy-and-hold's return has *negative* alpha — it earned its excess by owning more market.
`holds/yr` counts **completed holds** (`strategy.exit_causes`); vectorbt's `trades/yr` counts
*lots*, which every re-trim splits (the v26 overlay pick showed 210 lots/yr on 31 holds/yr).
`return ÷ vol` is the plain risk-reward number: a high-risk book is only worth running if it
beats buy-and-hold's.

In [ ]:
train = {label: score(p, panel, SCORE_FROM) for label, p in cands.items()}
for label, p in cands.items():
    ends = mod["exit_causes"](panel, **{**strat.params, **p})["exit"]
    train[label]["holds_per_year"] = (float((ends >= pd.Timestamp(SCORE_FROM, tz="UTC")).sum())
                                      / train[label]["years"])
COLS = {"cagr": PCT, "bench_cagr": PCT, "excess_cagr": PCT, "alpha_cagr": PCT, "beta": "{:.2f}",
        "volatility": "{:.1%}", "bench_volatility": "{:.1%}", "ret_per_vol": "{:.2f}",
        "bench_ret_per_vol": "{:.2f}", "max_drawdown": PCT, "holds_per_year": "{:.0f}",
        "trades_per_year": "{:.0f}"}
display(pd.DataFrame({label: {k: s[k] for k in COLS} for label, s in train.items()}).T
        .style.format(COLS))

### 3a · Costs and fill timing

Each candidate's train alpha at 1×/2×/3× the configured costs; the **round-trip cost** (quoted
spread plus slippage, in bps) at which that alpha reaches zero, found by bisection with no cap;
its alpha with the commission priced at **this account's real order size** (`ACCOUNT ÷ slots`);
and its alpha under **both** fill models — `open` (every order at the next bar's open, what a
market-on-open order gets) and `pessimistic` (every buy at the bar's high, every sell at its low,
benchmark included).

Two different things can erase a daily strategy's edge. **Fill timing**: pessimistic fills charge
about 1% per leg, so at 3–4 book turnovers a year they cost more than the whole signal — a stress
test, and a warning row in §6, not the yardstick. **The $1 minimum commission**: the engine
spreads it over a $10,000 order by default (1 bp), but a 30-slot book on a $30k account places
$1,000 orders (10 bp). `config.yaml`'s `backtest.order_notional` makes every number in both
notebooks pay that; this column re-prices it for the `ACCOUNT` set in §0, so a different account
size can be read off without a new study. The ✓ marks the fill model `config.yaml` ships.

Two columns ask what a **real account** does to that alpha (hard gates since v32). **Late fills**: live orders go in market-on-open, and the stored open *is* the first trade (§5e); an order placed by hand in the first half hour lands a median ~50 bp away, so this column charges every leg `late_fill_bps` more — always adversely, which is the stress. **Whole shares**: the engine trades fractional shares; this column sizes every order in whole shares at `ACCOUNT ÷ slots`, so each slot's remainder waits in cash and a name priced above its slot is never bought. (Stored prices are split-adjusted, so before a split the rounding is under-counted; the holdout reading in §5a is the one at near-real prices.)

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})


def hurdle(params, strategy=strat, cfg=settings):
    """Train alpha after beta (the study's hurdle), scored from optuna.score_from."""
    return bt.evaluate(strategy, panel, params, settings=cfg, score_from=SCORE_FROM)[HURDLE]


def breakeven_bps(params, tol=5.0):
    """Round-trip cost (quoted spread + slippage, bps) at which the train hurdle reaches zero."""
    c = settings.costs
    base = c.spread_bps + 2.0 * c.slippage_bps      # fees charge spread/2 + slippage per leg

    def alive(extra):
        return hurdle(params, cfg=with_costs(spread_bps=base + extra, slippage_bps=0.0)) > 0

    if not alive(0.0):
        return base
    lo, hi = 0.0, 50.0
    while alive(hi):
        lo, hi = hi, 2.0 * hi
        if hi > 50_000:
            return np.inf
    while hi - lo > tol:
        mid = (lo + hi) / 2.0
        lo, hi = (mid, hi) if alive(mid) else (lo, mid)
    return base + hi


c = settings.costs
REAL_SIZE = "real order size"
costs = {}
for label, p in cands.items():
    row = {f"{m:g}x costs": hurdle(p, cfg=with_costs(
        spread_bps=c.spread_bps * m, slippage_bps=c.slippage_bps * m,
        commission_per_share=c.commission_per_share * m)) for m in COST_MULTIPLES}
    row["breakeven round-trip bps"] = breakeven_bps(p)
    slot = ACCOUNT / {**strat.params, **p}["max_positions"]
    row[REAL_SIZE] = hurdle(p, strategy=replace(strat, order_notional=slot))
    row[LATE_COL] = hurdle(p, cfg=LATE_FILLS)
    row[WHOLE_COL] = hurdle(p, strategy=whole_shares(p), cfg=AT_ACCOUNT)
    for fill in ("open", "pessimistic"):
        row[f"fill: {fill}" + (" ✓" if fill == strat.fill else "")] = hurdle(
            p, strategy=replace(strat, fill=fill))
    costs[label] = row
costs = pd.DataFrame(costs).T
display(costs.style.format(PCT).format("{:.0f}", subset=["breakeven round-trip bps"]))
fees, slip = bt.strategy_costs(strat, panel, settings)
print(f"every cell is train {HURDLE}; the cost columns use the shipped fill ({strat.fill}) and "
      f"{fees * 1e4:.1f} bp fees + {slip * 1e4:.1f} bp slippage per leg "
      f"(order_notional ${strat.order_notional or 10_000:,.0f}); '{REAL_SIZE}' re-prices the "
      f"commission at ACCOUNT / slots = ${slot:,.0f} per order")

### 3b · Year by year (train)

Each calendar year of the scored training window on its own, warm, as **alpha after beta** and
as excess over buy-and-hold. An edge carried by one or two years is a regime bet, not a strategy;
excess well above alpha in a year means that year was won by beta.

In [ ]:
years = {}
for y in range(pd.Timestamp(SCORE_FROM).year, pd.Timestamp(d["end"]).year + 1):
    sub = slice_panel(panel, d["start"], f"{y}-12-31")        # history + the year
    start = max(f"{y}-01-01", SCORE_FROM)                      # never before the scored window
    for label, p in cands.items():
        st = bt.evaluate(strat, sub, p, settings=settings, score_from=start)
        years.setdefault(y, {})[(label, "alpha")] = st["alpha_cagr"]
        years[y][(label, "excess")] = st["excess_cagr"]
year_table = pd.DataFrame(years).T.rename_axis("train, per year")
display(year_table.style.format(PCT).background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3))
if not SCORE_FROM.endswith("01-01"):
    print(f"{pd.Timestamp(SCORE_FROM).year} is scored from {SCORE_FROM} only (an annualized "
          "part year)")

### 3d · Parameter stability — plateau or spike?

Nothing is searched since v28, so the neighbourhood is drawn over the **pinned book rules**
instead (`LOCAL_SPACE`: `q_out`, `q_in`, `max_hold`, `insider_window`) — a fixed value deserves the
same question as a picked one. (v27's version of this table was **empty**: `strategy.valid`
counts the composite weights, a candidate that carries only the searched keys has none, and
every neighbour was rejected — so its spike gate could not fail.)

Each knob moved one at a time by ±5% and ±10% of its own searched span (log-aware,
snapped to values the study could have suggested), re-scored on train **warm, on the study's
hurdle** (`alpha_cagr`) — one `bt.evaluate` per neighbour, because the column-stacked sweep can
neither score from `score_from` nor compute alpha. The summary counts the verdicts per
candidate; the `FOCUS` candidate's full table and tornado follow.

| verdict | meaning | action |
|---|---|---|
| `inert` | the neighbourhood scores the same | pin it; spend trials elsewhere (a local claim) |
| `plateau` | moves cost something, nothing breaks | fine |
| `cliff (low/high)` | one side loses half the edge in one step | take `suggest`, one step clear of the edge |
| `spike` | both sides collapse | **do not ship** — prefer a flatter front row |
| `off-center` | a neighbour beats it in sample | not an instruction: refer it to the walk-forward |
| `decisive` | (a categorical) the other choice loses the edge | keep it |

In [ ]:
LOCAL_SPACE = SPACE or {      # nothing searched: judge the pinned book rules instead
    "q_out": {"type": "float", "low": 0.60, "high": 0.85, "step": 0.05},
    "q_in": {"type": "float", "low": 0.85, "high": 0.95, "step": 0.05},
    "max_hold": {"type": "int", "low": 63, "high": 252, "log": True},
    "insider_window": {"type": "int", "low": 63, "high": 252, "log": True},
}


def warm_star(params):
    """opt.neighbors around a candidate, every neighbour scored warm by bt.evaluate.

    `strategy.valid` needs the WHOLE parameter set (it counts the composite weights), so every
    neighbour is validated on top of config.yaml's values, never as a bare partial dict.
    """
    center = {k: {**strat.params, **params}[k] for k in LOCAL_SPACE}
    star = opt.neighbors(center, LOCAL_SPACE,
                         valid=lambda c: strat.valid({**strat.params, **params, **c})
                         ).reset_index(drop=True)
    grid = [{**params, **{k: type(center[k])(row[k]) for k in center}}
            for row in star.to_dict("records")]
    metrics = pd.DataFrame([bt.evaluate(strat, panel, g, settings=settings, score_from=SCORE_FROM)
                            for g in grid])
    return pd.concat([star, metrics], axis=1)


stability = {}
for label, p in cands.items():
    star = warm_star(p)
    verdicts = opt.stability_report(star, metric=HURDLE,
                                    min_trades=FLOOR * train[label]["years"])
    center = float(star.loc[star["perturbed"] == "", HURDLE].iloc[0])
    near = star.loc[star["perturbed"] != "", HURDLE]
    stability[label] = {
        "neighbours": len(near), "alpha > 0": int((near > 0).sum()),
        "median keeps": near.median() / center if center > 0 else np.nan,
        "spikes": int((verdicts["verdict"] == "spike").sum()),
        "cliffs": int(verdicts["verdict"].str.startswith("cliff").sum()),
        "inert": int((verdicts["verdict"] == "inert").sum())}
    if label == FOCUS:
        focus_star, focus_verdicts = star, verdicts
display(pd.DataFrame(stability).T.style.format("{:.0f}").format("{:.0%}", subset=["median keeps"]))
print(f"\n{FOCUS}, knob by knob ({HURDLE}):")
display(focus_verdicts)
plots.plot_sensitivity(focus_star, metric=HURDLE).show()

### 3e · Shuffled books on train — is it the signal, or the construction? (information)

The same book — same slots, hysteresis, displacement, risk overlay, concentration cap and costs,
run through the engine's warm path — driven by a **symbol-shuffled** ranking: each simulation
hands every name another name's whole score history. The shuffle keeps everything about the
ranking except the one thing under test: how persistent it is, how often names cross the bands,
how much the book turns over — all identical — but a score no longer belongs to the stock whose
returns follow it. The **shuffled mean is what the construction alone earns** (costs, turnover,
holding 30 names instead of 500): read it as the drag every real book has to clear.

On train this is **information, not a gate**: the real ranking was chosen on this window and the
shuffles were not, so it passes easily even when the walk-forward fails (v26: z = +6 beside a
stitched out-of-sample alpha of −10.8%/yr). The gate is the same test on unseen data, §5d.

In [ ]:
def run_warm(signals_dict, on=None, start=None):
    """Any signal dict through the engine's warm path — bt._run_scored minus signals()."""
    on, start = (panel if on is None else on), (SCORE_FROM if start is None else start)
    window, sig = bt._score_window(on, signals_dict, start)
    fees, slip = bt.strategy_costs(strat, window, settings)
    pf = bt.run(window, sig, fees=fees, slippage=slip, init_cash=settings.backtest.init_cash,
                fill=strat.fill)
    st = bt.portfolio_stats(pf)
    st |= bt.active_stats(pf, window, fees=fees, slippage=slip, total_return=st["total_return"],
                          fill=strat.fill)
    st["returns"] = bt.equity_curve(pf)["value"].pct_change().fillna(0.0)
    st["bench_returns"] = bt.benchmark(window, fees=fees, slippage=slip, fill=strat.fill)[1]
    st["bench_cagr"] = st["cagr"] - st["excess_cagr"]
    return st


def book_run(parts, p, ranking, on=None, start=None):
    """The candidate's slots, risk overlay and retrim cap on any ranking -> run_warm's numbers.

    `on` must be the panel `parts` was composed from: `_exposure` reads its bars, and the window
    run reads its prices.
    """
    held = mod["_select"](dict(parts, score_pct=ranking), q_in=p["q_in"], q_out=p["q_out"],
                          max_hold=p["max_hold"], max_positions=p["max_positions"],
                          displace_margin=p["displace_margin"])["held"]
    weights = mod["_weights"](held, p["max_positions"])
    weights = weights.mul(mod["_exposure"](panel if on is None else on, weights, **p), axis=0)
    return run_warm({"weights": weights, "retrim_k": p["retrim_k"]}, on, start)


def book_hurdle(parts, p, ranking, on=None, start=None):
    """book_run's hurdle metric alone."""
    return book_run(parts, p, ranking, on, start)[HURDLE]


def shuffled(parts, rng, groups=None):
    """One symbol-shuffled ranking; `groups` (a per-symbol Series) keeps swaps inside a group."""
    rank, ids = parts["score_pct"], np.arange(parts["score_pct"].shape[1])
    order = ids.copy()
    labels = None if groups is None else groups.reindex(rank.columns).fillna(0).to_numpy()
    for g in ([None] if labels is None else np.unique(labels)):
        ix = ids if g is None else np.flatnonzero(labels == g)
        order[ix] = rng.permutation(ix)
    return (pd.DataFrame(rank.to_numpy()[:, order], index=rank.index, columns=rank.columns)
            .where(parts["eligible"]).rank(axis=1, pct=True))


# Liquidity tenths, fixed from the universe's own ranking window (the 126 bars before the first
# scored bar): shuffling INSIDE a tenth keeps every book's liquidity profile, so what is left is
# the choice among alike names. Used over the whole walk-forward (§4) and on the holdout (§5d).
ranking_window = (full.close * full.volume).loc[:pd.Timestamp(SCORE_FROM, tz="UTC")].tail(126)
TENTHS = np.ceil(ranking_window.median().rank(pct=True).fillna(0.5) * 10)

rng = np.random.default_rng(7)
PCTL = GATES["random_percentile"]
randoms = {}
for label, params in cands.items():
    p = {**strat.params, **params}
    parts = compose(panel, **p)
    real = book_hurdle(parts, p, parts["score_pct"])     # the same machinery on the REAL ranking
    sims = np.array([book_hurdle(parts, p, shuffled(parts, rng)) for _ in range(N_RANDOM_SIMS)])
    randoms[label] = {"real ranking": real, "shuffled mean": sims.mean(),
                      f"shuffled p{PCTL}": np.percentile(sims, PCTL),
                      "z": (real - sims.mean()) / sims.std() if sims.std() else np.nan}
    if abs(real - train[label][HURDLE]) > 1e-6:
        print(f"⚠ {label}: rebuilt book {real:+.4f} ≠ §3 {train[label][HURDLE]:+.4f} — the "
              "shuffle is not running the strategy's own machinery")
randoms = pd.DataFrame(randoms).T
display(randoms.style.format(PCT).format("{:+.1f}", subset=["z"]))

### 3f · Concentration — what the pinned `max_positions` trades away (train)

`max_positions` is pinned in `config.yaml` (30 since v27) as a **risk policy**, not searched: every study that
searched it answered "fewer slots" at whatever floor it was given (30 → 15 → 8 → 5 → 3 → 10),
because a narrower book of the top names is a purer — and far more fragile — bet. This table is
the record of that trade-off for each candidate at other book widths: alpha, the risk it costs
and the beta it takes on. Read it, do not pick from it — it is in-sample, and choosing the width
that scores best here is exactly the search the pin exists to stop.

In [ ]:
WIDTHS = sorted({15, 20, 30, 40, strat.params["max_positions"]})
concentration = {}
for label, p in cands.items():
    for width in WIDTHS:
        # the order size moves with the width, and with it the weight of the $1 minimum
        st = bt.evaluate(replace(strat, order_notional=ACCOUNT / width), panel,
                         {**p, "max_positions": width}, settings=settings, score_from=SCORE_FROM)
        concentration[(label, f"{width}" + (" ✓" if width == strat.params["max_positions"]
                                            else ""))] = {
            "alpha": st["alpha_cagr"], "excess": st["excess_cagr"], "beta": st["beta"],
            "volatility": st["volatility"], "max drawdown": st["max_drawdown"],
            "trades/yr": st["trades_per_year"]}
display(pd.DataFrame(concentration).T.rename_axis(["candidate", "slots"])
        .style.format(PCT).format("{:.2f}", subset=["beta"])
        .format("{:.1%}", subset=["volatility"]).format("{:.0f}", subset=["trades/yr"]))
print("✓ = the width config.yaml ships")

## 4 · Out of sample — the walk-forward of the whole procedure

A walk-forward tests the **procedure**, not a parameter set. Since v28 the procedure has no
optimizer in it; what it *does* have is a decision made from data — **which sleeves ship**. v27
made that decision once, on the 2020-07 → 2024-12 train window, and then walked forward through
2022–2024: periods the decision had already seen. So this section now repeats the decision
**inside every period**: every `WF_STEP_MONTHS` from `WF_START` it re-reads the §2a signal rule on
the data before that date only, switches on the sleeves that pass (equal weight; if none passes
the book holds the equal-weight universe — buy-and-hold, alpha zero by construction — since v32;
until v31 it sat in cash, a bet against the market), and trades the fixed book rules on the unseen period that follows — warm,
each period's panel **loaded on its own window**. *If I had followed this recipe from the start,
what would I have earned?* That is the **rule arm**, and the gates in §6 read it.

Beside it, the same periods for four fixed alternatives, each stitched and bootstrapped the same
way — read them as controls, never as a menu to pick from:

- **config sleeves, fixed** — what `config.yaml` ships, in every period (v27's "no tuning" arm;
  in-sample for the sleeve choice before 2025).
- **all five sleeves, equal weight** — no selection at all: does choosing sleeves earn its place?
- **ratings back in** — v27-v31's two sleeves: what the t ≥ 3 bar (v32) gives up or saves.
- **control: the rule without its calendar-year test** — added after v30 exposed a knife-edge in
  the pre-registered rule: "IC positive in ≥ 80% of calendar years" fails a sleeve for one year at
  −0.000, and with two to four years of history that one year locks it out for years (on
  `Liquid1000` it kept `short_interest` out of five periods). This arm re-reads the rule on t and
  the dollar-volume groups alone. It was designed after looking at a walk-forward, so it is a
  **control, never the gate**: if it clearly beats the rule arm, pre-register a better rule for
  the *next* study rather than swapping it in here.
- **config sleeves, sell at rank 0.47** — the v26 habit of holding a name down to the middle of
  the ranking.

**Shuffled books over the whole walk-forward** (since v29). In every period the rule arm's book is
also run on `N_WF_SIMS` symbol-shuffled rankings, shuffled only **inside dollar-volume tenths**, and
each simulation is stitched across the periods exactly like the real record. That is §5d's test on
4.7 years instead of 1.7, for the arm that never saw its test data — the strongest skill test in
this notebook, and a hard gate. The shuffles pay the same costs, run the same rules and keep the
same liquidity profile, so *real minus shuffled mean* is alpha against a liquidity-matched book.

**The naive basket.** An equal-weight buy-and-hold of the **50 most traded names**, re-chosen at
every period start from the 126 bars before it: in 2025-26 that basket beat the book, so it runs
beside every arm as the honest "did you just own the popular names?" control.

**§4c — what earned what.** Two sets of fixed arms, scored and stitched like the rest: the **road**
from the v26 pick to today's rules, one change at a time, and **flip-backs** — today's rules with
one choice reversed — including the two real-account stresses, **late fills** and **whole
shares** at `ACCOUNT`.

It runs to the **last bar**; periods from 2025 on overlap the holdout, which earlier study versions
have read, and are marked *seen before*. A last period shorter than `WF_MIN_BARS` joins the
stitched record but does not count as a period.

In [ ]:
origins = pd.date_range(WF_START, LAST, freq=f"{WF_STEP_MONTHS}MS")
FOLDS = [(str((o - pd.Timedelta(days=1)).date()), str(o.date()),
          min(str((o + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)).date()), LAST))
         for o in origins]
OFF = {w: 0.0 for w in WEIGHT_OF.values()}
RULE_ARM, FIXED_ARM = "sleeve rule re-read each period", "config sleeves, fixed"
ALT_ARM = "control: the rule without its calendar-year test"
BASE_ARM = f"baseline: {BASELINE_LABEL} (rule re-read)"
BASE_FIXED = f"baseline: {BASELINE_LABEL}, fixed"
BASE_LATE = f"baseline: {BASELINE_LABEL}, fixed, {LATE_COL}"
NAIVE_ARM, NAIVE_N = "naive: the 50 most traded names", 50
TUNED_ARM = "rule arm + the search re-run each period"
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}


def as_params(row):
    """A trial row's searched parameters, cast back to the types signals() expects."""
    return {k: next(c for c in SPACE[k]["choices"] if c == row[k])
            if SPACE[k]["type"] == "categorical"
            else row[k] if isinstance(row[k], str)
            else int(round(float(row[k]))) if k in SPACE_INT else round(float(row[k]), 10)
            for k in SPACE}


def retune(train_panel, weights, tr1):
    """The study's own search, re-run on ONE training window with the sleeves the rule chose there.

    Same space, sampler, trial budget (WF_TRIALS) and picking rule as the main study, so the tuned
    arm tests the optimizer exactly as it made the pick. Returns (params, convergence).
    """
    how = replace(strat, params={**strat.params, **weights})
    name = f"{STUDY}-wf-{tr1}-" + "+".join(sorted(k for k, w in weights.items() if w > 0))
    try:
        done = len(opt.load_study(how, name).trials)
    except KeyError:
        done = 0
    study = opt.run_study(how, train_panel, settings=settings, study_name=name,
                          n_trials=max(0, WF_TRIALS - done), score_from=SCORE_FROM)
    trials = opt.trials_frame(study)
    pool = trials[trials["trades_per_year"] >= FLOOR]
    best = trials.sort_values("trial")[HURDLE].cummax().to_numpy()
    gain = float(best[-1] - best[max(0, int(len(best) * 2 / 3) - 1)])
    return as_params(opt.pick_from_front(pool if len(pool) else trials)), gain


def sized(slots, **overrides):
    """The strategy with its commission priced for `slots` positions of ACCOUNT."""
    return replace(strat, order_notional=ACCOUNT / slots, **overrides)


# The v26 pick (trial 3016), as it ran: 5 tuned sleeves, a 0.73/0.47 band, 15 slots, a vol_target
# overlay at its floor, pessimistic fills and the $10,000-order commission.
V26_PICK = {"insider_window": 117, "rating_halflife": 23, "ptrev_halflife": 8, "w_insider": 1.0,
            "w_rating": 0.5, "w_ptrev": 0.25, "w_dtc": 1.0, "w_recency": 1.25, "q_in": 0.73,
            "q_out": 0.47, "max_hold": 89, "displace_margin": 0.72, "max_positions": 15,
            "risk_method": "vol_target", "risk_off_exposure": 0.7, "target_vol": 0.10,
            "neutral_groups": 0}
TODAY = dict(strat.params)
SLEEVES = {k: TODAY[k] for k in WEIGHT_OF.values()}
BAND = {k: TODAY[k] for k in ("q_in", "q_out", "max_hold", "displace_margin", "insider_window",
                              "rating_halflife", "ptrev_halflife")}
NO_OVERLAY = {**V26_PICK, "risk_method": "none"}
ROAD = {        # from the v26 pick to today's rules, one change at a time (last step: FIXED_ARM)
    "road 0 · the v26 pick as it ran": (V26_PICK, replace(strat, fill="pessimistic",
                                                          order_notional=None)),
    "road 1 · + fills at the open, commission at the real order size": (V26_PICK, sized(15)),
    "road 2 · + risk overlay off": (NO_OVERLAY, sized(15)),
    "road 3 · + sleeves chosen by the signal rule": ({**NO_OVERLAY, **SLEEVES}, sized(15)),
    "road 4 · + sell when a name leaves the top fifth": ({**NO_OVERLAY, **SLEEVES, **BAND},
                                                         sized(15)),
    "road 5 · + 30 slots": ({**NO_OVERLAY, **SLEEVES, **BAND,
                                     "max_positions": TODAY["max_positions"]}, strat),
}
FLIPS = {       # today's rules with ONE choice reversed
    "flip · all five sleeves, equal weight": (ALL_ON, strat),
    "flip · ratings back in (v31's sleeves)": ({"w_rating": 1.0}, strat),
    "flip · sell at rank 0.47": ({"q_out": 0.47}, strat),
    "flip · 15 slots": ({"max_positions": 15}, sized(15)),
    "flip · vol_target overlay at 0.10": ({"risk_method": "vol_target", "target_vol": 0.10}, strat),
    "flip · pessimistic fills": ({}, replace(strat, fill="pessimistic")),
    "flip · plain ranking (the liquidity lean back in)": ({"neutral_groups": 0}, strat),
    # the real account (blockers 2 and 3): a third element is the settings the arm runs under
    f"flip · {LATE_COL}": ({}, strat, LATE_FILLS),
    f"flip · {WHOLE_COL}": ({}, whole_shares({}), AT_ACCOUNT),
}
CONTROLS = {FIXED_ARM: ({}, strat), **FLIPS, **ROAD}



def rule_sleeves(train_panel, params=None, gates=None, mask=None):
    """The §2a signal rule read on ONE training window -> [(weights, sleeve names)] x 2.

    First as pre-registered; then without its calendar-year test, for the control arm.
    `params` / `gates` read it as another version would (the §4d baseline).
    """
    saved = dict(GATES)
    GATES.update(gates or {})
    try:
        summary = fair_check(train_panel, {**(params or {}), **ALL_ON},
                             SCORE_FROM, mask=mask)[0].drop("COMPOSITE")
        readings = []
        for years_test in (True, False):
            passed = [k for k, row in summary.iterrows() if signal_rule(row, years_test)[0]]
            readings.append((OFF | {WEIGHT_OF[k]: 1.0 for k in passed}, passed))
    finally:
        GATES.clear()
        GATES.update(saved)
    return readings


def contributions(s, window):
    """Each name's summed active contribution in one scored period: w[t-1] x (r[t] - bench[t]).

    The attribution behind the §4d leave-top-5-out check: a book whose alpha is a handful of
    lottery winners (NVDA alone was ~1.8 points of v34's train alpha) can "improve" by landing on
    one more of them, which a time bootstrap cannot see.
    """
    w = bt._book_weights(s["pf"]).shift(1).fillna(0.0)
    r = window.close.loc[w.index].pct_change().fillna(0.0)
    return w.mul(r.sub(s["bench_returns"].reindex(w.index).fillna(0.0), axis=0)).sum()


def naive_weights(window, te0):
    """Equal-weight buy-and-hold of the NAIVE_N most traded names, ranked on 126 bars before te0."""
    dollars = (window.close * window.volume).loc[:pd.Timestamp(te0, tz="UTC")].iloc[:-1].tail(126)
    w = pd.DataFrame(0.0, index=window.close.index, columns=window.close.columns)
    w[dollars.median().nlargest(NAIVE_N).index] = 1.0 / NAIVE_N
    return w


def stitched(pairs):
    """One continuous out-of-sample record from per-period (book, buy-and-hold) daily returns."""
    r, b = pd.concat([x for x, _ in pairs]), pd.concat([y for _, y in pairs])
    years = bt._years(r.index)
    cagr, bench_cagr, beta, alpha = (float(x[0]) for x in alpha_of(r.to_numpy(), b.to_numpy(),
                                                                   years))
    ann = np.sqrt(len(r) / years)
    return {"years": years, "cagr": cagr, "buy & hold": bench_cagr, "beta": beta, "alpha": alpha,
            **bootstrap(r.to_numpy(), b.to_numpy(), years),
            "return / vol": cagr / (r.std() * ann) if r.std() else 0.0,
            "B&H return / vol": bench_cagr / (b.std() * ann)}


rng_wf = np.random.default_rng(23)
wf, arm_returns, arm_alpha, carried, refits = [], {}, {}, {}, []
CONTRIB = {}                                   # arm -> summed active contribution per name
sim_returns = [[] for _ in range(N_WF_SIMS)]
for tr1, te0, te1 in FOLDS:
    # The previous period's test panel ends exactly where this period's training ends — reusable
    # only while every period trades the same names.
    period_syms = members(te0) if WF_UNIVERSE == "rolling" else None
    train_panel = carried.pop(tr1, None) if WF_UNIVERSE == "fixed" else None
    rule_mask = None
    if WF_UNIVERSE == "rolling":            # the rule reads every past membership, point in time
        history = pit_history(te0)
        train_panel = load(tr1, sorted(set().union(*history.values())))
        rule_mask = member_mask(train_panel, history)
    elif train_panel is None:
        train_panel = load(tr1, period_syms)
    (weights, passed), (weights_alt, passed_alt) = rule_sleeves(train_panel, mask=rule_mask)
    is_alpha = (bt.evaluate(strat, train_panel, weights, settings=settings,
                            score_from=SCORE_FROM)[HURDLE] if passed else 0.0)
    window = load(te1, period_syms)             # history + the test period, on its own load
    carried = {te1: window} if WF_UNIVERSE == "fixed" else {}
    period = f"{te0[:7]}..{te1[:7]}"
    scores = {arm: score(spec[0], window, te0, strategy=spec[1],
                         cfg=spec[2] if len(spec) > 2 else None)
              for arm, spec in CONTROLS.items()}
    scores[NAIVE_ARM] = run_warm({"weights": naive_weights(window, te0)}, window, te0)
    ref = scores[FIXED_ARM]
    if passed:                                  # else: no sleeve passed -> hold buy-and-hold
        scores[RULE_ARM] = score(weights, window, te0)
    if passed_alt:
        scores[ALT_ARM] = (scores[RULE_ARM] if passed_alt == passed
                           else score(weights_alt, window, te0))
    if SPACE and passed:                        # the optimizer, re-run on this period's training
        tuned, gain = retune(train_panel, weights, tr1)
        scores[TUNED_ARM] = score({**weights, **tuned}, window, te0)
        refits.append({"period": period, **tuned, "last third added": gain,
                       "OOS alpha tuned": scores[TUNED_ARM]["alpha_cagr"],
                       "OOS alpha untuned": scores[RULE_ARM]["alpha_cagr"]})
    if BASELINE_LABEL:                          # the adopted version, read and traded its own way
        same_names = BASELINE_WF_UNIVERSE == WF_UNIVERSE
        base_train = train_panel if same_names else load(tr1)
        base_window = window if same_names else load(te1)
        (base_w, base_passed), _ = rule_sleeves(base_train, BASELINE, BASELINE_GATES,
                                                mask=rule_mask if same_names else None)
        scores[BASE_ARM] = (score({**BASELINE, **base_w}, base_window, te0) if base_passed
                            else None)
        scores[BASE_FIXED] = score(BASELINE, base_window, te0)
        scores[BASE_LATE] = score(BASELINE, base_window, te0, cfg=LATE_FILLS)
        if not base_passed:                     # the baseline held ITS buy & hold that period
            fallback = score({}, base_window, te0)
            scores[BASE_ARM] = {"returns": fallback["bench_returns"],
                                "bench_returns": fallback["bench_returns"], "alpha_cagr": 0.0}
    for arm in (RULE_ARM, *([BASE_ARM] if BASELINE_LABEL else [])):
        if scores.get(arm) and "pf" in scores[arm]:
            CONTRIB[arm] = CONTRIB.get(arm, pd.Series(dtype=float)).add(
                contributions(scores[arm], window if arm == RULE_ARM else base_window),
                fill_value=0.0)
    for arm in (RULE_ARM, ALT_ARM, *CONTROLS, NAIVE_ARM,
                *([BASE_ARM, BASE_FIXED, BASE_LATE] if BASELINE_LABEL else []),
                *([TUNED_ARM] if SPACE else [])):
        s = scores.get(arm)
        arm_returns.setdefault(arm, []).append(
            (s["returns"], s["bench_returns"]) if s
            else (ref["bench_returns"], ref["bench_returns"]))   # no sleeve: the benchmark itself
        arm_alpha.setdefault(arm, {})[period] = s["alpha_cagr"] if s else 0.0
    # the rule arm's own book on shuffled rankings, inside liquidity tenths, this period
    if passed and N_WF_SIMS:
        p_rule = {**strat.params, **weights}
        parts = compose(window, **p_rule)
        for sim in sim_returns:
            sim.append(book_run(parts, p_rule, shuffled(parts, rng_wf, TENTHS), window,
                                te0)["returns"])
    else:
        for sim in sim_returns:
            sim.append(ref["bench_returns"])
    oos = scores.get(RULE_ARM)
    wf.append({"period": period, "bars": len(ref["returns"]), "seen before": te0 >= HOLDOUT_START,
               "names": len(window.symbols),
               "in data.universe": len(set(window.symbols) & set(syms)),
               "sleeves the rule chose": " + ".join(passed) or "— none: buy & hold —",
               "without the year test": " + ".join(passed_alt) or "— none: buy & hold —",
               **({"baseline chose": " + ".join(base_passed) or "— none: buy & hold —"}
                  if BASELINE_LABEL else {}),
               f"IS {HURDLE}": float(is_alpha), "OOS alpha": oos["alpha_cagr"] if oos else 0.0,
               "OOS excess": oos["excess_cagr"] if oos else 0.0,
               "OOS beta": oos["beta"] if oos else 1.0,
               "OOS volatility": oos["volatility"] if oos else ref["bench_volatility"],
               "OOS trades": int(oos["n_trades"]) if oos else 0})
    del scores, ref, oos                        # each score() holds a whole vectorbt portfolio

wf_table = pd.DataFrame(wf).set_index("period")
if refits:
    REFITS = pd.DataFrame(refits).set_index("period")
    print(f"the search re-run in every period ({WF_TRIALS} trials each) — what it picked, how much "
          "its last third of trials still added (convergence), and what the pick earned:")
    display(REFITS.style.format({"last third added": "{:+.2%}", "OOS alpha tuned": PCT,
                                 "OOS alpha untuned": PCT}))
display(wf_table.style.format({f"IS {HURDLE}": PCT, "OOS alpha": PCT, "OOS excess": PCT,
                               "OOS beta": "{:.2f}", "OOS volatility": "{:.1%}"}))

# --- the stitched out-of-sample record, one row per arm ----------------------------------------
WF_ARMS = pd.DataFrame({arm: stitched(pairs) for arm, pairs in arm_returns.items()}).T
WF_STITCHED = WF_ARMS.loc[RULE_ARM].to_dict()
first, last = arm_returns[RULE_ARM][0][0].index[0], arm_returns[RULE_ARM][-1][0].index[-1]
CORE = [RULE_ARM, *([TUNED_ARM] if SPACE else []), ALT_ARM, FIXED_ARM,
        "flip · all five sleeves, equal weight", "flip · ratings back in (v31's sleeves)",
        NAIVE_ARM]


def show_arms(frame):
    """A table of stitched arms: percents, except the ratios (2 decimals) and the probabilities."""
    ratios = [c for c in ("years", "beta", "return / vol", "B&H return / vol") if c in frame]
    probs = [c for c in ("P(alpha > 0)", "P(excess > 0)") if c in frame]
    display(frame.style.format(PCT, na_rep="—").format("{:.2f}", subset=ratios)
            .format("{:.0%}", subset=probs))


print(f"stitched out of sample, {first.date()} → {last.date()} · SPY over the same window "
      f"{window_cagr(SPY, first.date(), last.date()):+.1%}/yr:")
show_arms(WF_ARMS.loc[CORE])
full_periods = wf_table[wf_table["bars"] >= WF_MIN_BARS]
is_x, oos_x = full_periods[f"IS {HURDLE}"].to_numpy(), full_periods["OOS alpha"].to_numpy()
WF_POSITIVE = int((oos_x > 0).sum())
t, p_value = stats.ttest_rel(is_x, oos_x)
print(f"rule arm: mean IS {HURDLE} {is_x.mean():+.2%}/yr vs mean OOS alpha {oos_x.mean():+.2%}/yr "
      f"(paired t={t:.2f}, p={p_value:.2f}) · {WF_POSITIVE}/{len(oos_x)} full periods with "
      "alpha > 0")
chosen = pd.Series([k for w in wf for k in w["sleeves the rule chose"].split(" + ")
                    if not k.startswith("—")]).value_counts()
SLEEVE_VOTES = " · ".join(f"{k} {n}/{len(wf)}" for k, n in chosen.items()) or "none, ever"
print(f"how often the rule switched each sleeve on: {SLEEVE_VOTES}")

# --- the rule arm against its own book on shuffled rankings, over the whole walk-forward --------
bench_all = pd.concat([b for _, b in arm_returns[RULE_ARM]]).to_numpy()
sim_alpha = np.array([float(alpha_of(pd.concat(sim).to_numpy(), bench_all,
                                     WF_STITCHED["years"])[3][0]) for sim in sim_returns])
WF_SHUFFLE = {"real": WF_STITCHED["alpha"], "shuffled mean": float(sim_alpha.mean()),
              f"shuffled p{PCTL}": float(np.percentile(sim_alpha, PCTL)),
              "real − mean": WF_STITCHED["alpha"] - float(sim_alpha.mean()),
              "shuffles ≥ real": float((sim_alpha >= WF_STITCHED["alpha"]).mean())}
print(f"\nrule arm vs {N_WF_SIMS} shuffled books (inside dollar-volume tenths), stitched alike:")
display(pd.DataFrame({RULE_ARM: WF_SHUFFLE}).T.style.format(PCT)
        .format("{:.0%}", subset=["shuffles ≥ real"]))

### 4b · Every arm, period by period

Out-of-sample alpha of each arm in each period (the stitched table above is the verdict; this shows
where it came from). The counts say how often the rule arm beat each control over the full
periods — an arm that wins on the mean but loses most periods is living on one or two half-years.

In [ ]:
by_period = pd.DataFrame(arm_alpha).reindex(wf_table.index)
display(by_period[CORE].style.format(PCT).background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3))
al = by_period.loc[full_periods.index]
for arm in CORE[1:]:
    print(f"rule arm beats '{arm}' in {int((al[RULE_ARM] > al[arm]).sum())}/{len(al)} full periods "
          f"(mean alpha {al[RULE_ARM].mean():+.1%} vs {al[arm].mean():+.1%})")

### 4c · What earned what — the road from v26, and today's rules with one choice reversed

v27 changed every axis at once, so its result could not say which change earned what. These are
fixed parameter sets (no refits, no rule re-reading), scored on the same ten periods and stitched the
same way. **The road** starts at the v26 pick exactly as it ran and adds one change per step until
it arrives at what `config.yaml` ships today; the order matters a little (the steps interact), so
the **flip-backs** ask the same question the other way round: today's rules with ONE choice
reversed. Where both agree, the attribution is solid.

Caveat, the same as for the `config sleeves, fixed` arm: steps that use today's sleeve choice saw
2022–2024 when that choice was made. Read the *differences between rows*, which share that flaw
equally, rather than any level.

In [ ]:
SHOW = ["cagr", "buy & hold", "beta", "alpha", "P(alpha > 0)", "return / vol"]
road = WF_ARMS.loc[[*ROAD, FIXED_ARM], SHOW].rename(
    index={FIXED_ARM: "road 6 · + liquidity-neutral ranking = config.yaml today"})
road.insert(4, "step adds", road["alpha"].diff())
print("THE ROAD — stitched walk-forward, each row = the row above plus one change:")
show_arms(road)
flips = WF_ARMS.loc[[FIXED_ARM, *FLIPS], SHOW]
flips.insert(4, "costs vs today", flips["alpha"] - flips.loc[FIXED_ARM, "alpha"])
print("FLIP-BACKS — today's rules with one choice reversed (negative = today's choice earns that):")
show_arms(flips)

### 4d · Better than the adopted version? (paired)

Every version from v34 on is compared with the **last adopted version**, re-run inside this same
walk-forward under today's code and data (`BASELINE` in §0). Two separate 90% intervals overlap
even when one book beats the other on most days, because both carry the same market and many of
the same names; so the two daily return series are **bootstrapped together, on the same blocks**,
and the table reports the difference in alpha after beta, its interval and P(Δ > 0) — for the rule
arm (the procedure) and for the fixed rules — plus how many full periods each side won, and the
same difference on TRAIN. A change that helps out of sample but hurts in-sample (or the reverse)
is a coin flip read twice, not an improvement.

In [ ]:
VS_BASELINE, VS_UNTUNED, LEAVE5 = {}, {}, {}
if SPACE:
    r_t = pd.concat([x for x, _ in arm_returns[TUNED_ARM]])
    r_u, b_u = (pd.concat([x for x, _ in arm_returns[RULE_ARM]]),
                pd.concat([b for _, b in arm_returns[RULE_ARM]]))
    VS_UNTUNED = paired(r_t.to_numpy(), r_u.to_numpy(), b_u.to_numpy(), WF_STITCHED["years"])
    per = by_period.loc[full_periods.index]
    VS_UNTUNED["periods won"] = f"{int((per[TUNED_ARM] > per[RULE_ARM]).sum())}/{len(per)}"
    print("does the optimizer earn its place? the re-tuned rule arm against the untuned one, "
          "paired:")
    display(pd.DataFrame({"tuned − untuned": VS_UNTUNED}).T.style
            .format(PCT, subset=["Δ alpha", "Δ p5", "Δ p95"]).format("{:.0%}", subset=["P(Δ > 0)"]))
if BASELINE_LABEL:
    def stitched_series(arm):
        return (pd.concat([x for x, _ in arm_returns[arm]]),
                pd.concat([b for _, b in arm_returns[arm]]))

    rows = {}
    # PRIMARY = the fixed rules (same sleeves in every period: the mechanism alone); CLEAN = the
    # same on 2025+ only, the one stretch no a-priori choice was read from; LATE = PRIMARY with
    # every leg 50 bp worse than the open on BOTH sides. Each side is scored against its OWN market.
    for label, (mine, theirs) in {"rule arm (procedure)": (RULE_ARM, BASE_ARM),
                                  "fixed rules (PRIMARY)": (FIXED_ARM, BASE_FIXED),
                                  "fixed rules, late fills (LATE)": (f"flip · {LATE_COL}",
                                                                     BASE_LATE)}.items():
        (r, bench), (base, base_b) = stitched_series(mine), stitched_series(theirs)
        rows[label] = paired(r.to_numpy(), base.to_numpy(), bench.to_numpy(),
                             WF_STITCHED["years"], base_bench=base_b.to_numpy())
        if label.startswith("fixed rules (PRIMARY)"):
            late = r.index >= pd.Timestamp(HOLDOUT_START, tz="UTC")
            rows["fixed rules, 2025+ only (CLEAN)"] = paired(
                r[late].to_numpy(), base[late].to_numpy(), bench[late].to_numpy(),
                bt._years(r[late].index), base_bench=base_b[late].to_numpy())
        per = by_period.loc[full_periods.index] if mine in by_period else None
        if per is not None and theirs in by_period:
            rows[label]["periods won"] = f"{int((per[mine] > per[theirs]).sum())}/{len(per)}"
            if label.startswith("fixed rules (PRIMARY)"):
                clean = per.loc[per.index >= "2025"]
                rows["fixed rules, 2025+ only (CLEAN)"]["periods won"] = (
                    f"{int((clean[mine] > clean[theirs]).sum())}/{len(clean)}")
    # (a) the rule arm on the periods where BOTH rules chose the same sleeves: the mechanism alone.
    #     A period where the on/off decision flipped moves a whole half-year between buy & hold and
    #     the sleeve book, far more than any mechanism, and its sign is mostly known in advance.
    same = [k for k, row in enumerate(wf) if row["sleeves the rule chose"] == row["baseline chose"]]
    flips = [row["period"] for row in wf if row["sleeves the rule chose"] != row["baseline chose"]]
    per_bar_years = WF_STITCHED["years"] / sum(len(x) for x, _ in arm_returns[RULE_ARM])
if BASELINE_LABEL and same:
    r_same = pd.concat([arm_returns[RULE_ARM][k][0] for k in same])
    b_same = pd.concat([arm_returns[BASE_ARM][k][0] for k in same])
    m_same = pd.concat([arm_returns[RULE_ARM][k][1] for k in same])
    bm_same = pd.concat([arm_returns[BASE_ARM][k][1] for k in same])
    rows["rule arm, same-decision periods"] = paired(
        r_same.to_numpy(), b_same.to_numpy(), m_same.to_numpy(), len(r_same) * per_bar_years,
        base_bench=bm_same.to_numpy())
    traded = [k for k in same if wf[k]["period"] in full_periods.index
              and not wf[k]["sleeves the rule chose"].startswith("—")]
    won = sum(arm_alpha[RULE_ARM][wf[k]["period"]] > arm_alpha[BASE_ARM][wf[k]["period"]]
              for k in traded)
    rows["rule arm, same-decision periods"]["periods won"] = f"{won}/{len(traded)}"
if BASELINE_LABEL:
    # (b) leave the top-5 contributing names of EITHER book out: active return per year, both arms
    contrib = {arm: CONTRIB.get(arm, pd.Series(dtype=float)) for arm in (RULE_ARM, BASE_ARM)}
    top = set(contrib[RULE_ARM].nlargest(5).index) | set(contrib[BASE_ARM].nlargest(5).index)
    active = {arm: (c.sum() / WF_STITCHED["years"],
                    c.drop(list(top), errors="ignore").sum() / WF_STITCHED["years"])
              for arm, c in contrib.items()}
    LEAVE5 = {"names left out": ", ".join(sorted(top)),
              "Δ active, all names": active[RULE_ARM][0] - active[BASE_ARM][0],
              "Δ active, without them": active[RULE_ARM][1] - active[BASE_ARM][1]}
    base_train = hurdle(BASELINE) if BASELINE else hurdle({})
    for label in rows:
        rows[label]["Δ train alpha"] = train[FOCUS]["alpha_cagr"] - base_train
    VS_BASELINE = rows
    print(f"{STUDY} against {BASELINE_LABEL} — alpha after beta, same walk-forward, same days:")
    display(pd.DataFrame(rows).T.style.format(PCT, subset=["Δ alpha", "Δ p5", "Δ p95",
                                                           "Δ train alpha"])
            .format("{:.0%}", subset=["P(Δ > 0)"]))
    print(f"periods where the two rules decided differently (excluded from the same-decision row): "
          f"{', '.join(flips) or 'none'}")
    print(f"leave-top-5-out (arithmetic active return per year, rule arms): "
          f"Δ {LEAVE5['Δ active, all names']:+.1%} with every name → "
          f"{LEAVE5['Δ active, without them']:+.1%} without {LEAVE5['names left out']}")
else:
    print("no BASELINE set in §0 — nothing to compare against")

## 5 · The holdout — used, so read it as a warning light

2025+ is never loaded by the optimization notebook — but it is **not untouched**: this window has
been read under eight study versions (v13 → v26), and design changes followed those reads. A
candidate is scored on it **warm**, and everything measured on it alone is a **warning** in §6,
not a gate. Two reasons: a result on a window the design has already seen proves less than it
looks, and ~1.7 years of a 30-name book is a wide interval in any case — a truly good book of
this kind would fail a "90% sure" test here more often than not. The exception is §5d, which is
a *relative* test and stays a gate.

The holdout grows every time new bars are ingested, so its end date is printed with the table;
annualizing a part year exaggerates in both directions, so each calendar year is also shown.

### 5a · Against buy-and-hold

In [ ]:
hold, rows = {}, {}
HOLD_YEARS = range(pd.Timestamp(HOLDOUT_START).year, pd.Timestamp(LAST).year + 1)
for label, p in cands.items():
    hold[label] = h = score(p, full, HOLDOUT_START)
    h |= bootstrap(h["returns"].to_numpy(), h["bench_returns"].to_numpy(), h["years"])
    h["year_alpha"] = {}
    rows[label] = {"CAGR": h["cagr"], "buy & hold": h["bench_cagr"], "excess": h["excess_cagr"],
                   "alpha after beta": h["alpha_cagr"], "P(alpha > 0)": h["P(alpha > 0)"],
                   "alpha p5": h["alpha p5"], "alpha p95": h["alpha p95"], "beta": h["beta"],
                   "IR": h["info_ratio"], "volatility": h["volatility"],
                   "B&H volatility": h["bench_volatility"], "return / vol": h["ret_per_vol"],
                   "B&H return / vol": h["bench_ret_per_vol"],
                   "max drawdown": h["max_drawdown"], "trades/yr": h["trades_per_year"]}
    # The commission this book's own orders would have cost at ACCOUNT scale, exactly:
    # max($1, shares x per-share) per order, as a share of the account per year.
    orders, scale = h["pf"].orders.records_readable, ACCOUNT / settings.backtest.init_cash
    commission = np.maximum(settings.costs.min_commission,
                            orders["Size"] * scale * settings.costs.commission_per_share).sum()
    rows[label]["commission / yr"] = (commission / (scale * float(h["pf"].value().mean()))
                                      / h["years"])
    rows[label]["orders / yr"] = len(orders) / h["years"]
    h["alpha_late"] = score(p, full, HOLDOUT_START, cfg=LATE_FILLS)["alpha_cagr"]
    h["alpha_whole"] = score(p, full, HOLDOUT_START, strategy=whole_shares(p),
                             cfg=AT_ACCOUNT)["alpha_cagr"]
    rows[label][f"alpha · {LATE_COL}"] = h["alpha_late"]
    rows[label][f"alpha · {WHOLE_COL}"] = h["alpha_whole"]
    for y in HOLD_YEARS:
        sub = slice_panel(full, d["start"], f"{y}-12-31")
        if len(sub.close.loc[f"{y}-01-01":]) >= 40:
            st = bt.evaluate(strat, sub, p, settings=settings, score_from=f"{y}-01-01")
            h["year_alpha"][y] = st["alpha_cagr"]
            rows[label][f"alpha {y}"] = st["alpha_cagr"]
            rows[label][f"excess {y}"] = st["excess_cagr"]
hold_table = pd.DataFrame(rows).T
display(hold_table.style.format(PCT).format("{:+.2f}", subset=["IR", "beta"])
        .format("{:.0%}", subset=["P(alpha > 0)"])
        .format("{:.1%}", subset=["volatility", "B&H volatility"])
        .format("{:.2f}", subset=["return / vol", "B&H return / vol"])
        .format("{:.2%}", subset=["commission / yr"])
        .format("{:.0f}", subset=["trades/yr", "orders / yr"]))
# exit_causes counts HOLDS; bt.trade_records counts vectorbt lots, which every retrim splits.
n_holds = {label: int((mod["exit_causes"](full, **{**strat.params, **p})["exit"]
                       >= pd.Timestamp(HOLDOUT_START, tz="UTC")).sum())
           for label, p in cands.items()}
print(f"holdout {HOLDOUT_START} → {LAST} ({hold[FOCUS]['years']:.2f} years) · completed "
      f"holds: {n_holds} — the sample every holdout number stands on")

### 5b · Does the ranking still select stocks on the holdout?

The §2a tables again, on the holdout (the composite is built on the full history and scored from
the holdout start): the plain IC, then the fair check — beta-adjusted, by dollar-volume group and
by fifth of the ranking — and §2d's held-vs-ranking table. Positive on **both** windows = a sleeve
carrying real selection; positive in train and ~0 or negative here is the trap this strategy has
fallen into before. The last table puts the composite's **fair IC per calendar year** side by
side, train years from the train panel and 2025+ from the full one.

Read the 3- and 6-month rows lightly: on ~420 bars they rest on two or three independent windows,
and a Newey–West t with a lag that long is not reliable. A warning in §6, never a reason to
re-weight a sleeve — choosing weights from these rows would burn the window again.

In [ ]:
hold_ic = {label: sleeve_table(full, p, start=HOLDOUT_START) for label, p in cands.items()}
show_ic(hold_ic, "HOLDOUT")
fair_hold = {label: fair_check(full, p, HOLDOUT_START) for label, p in cands.items()}
show_fair(fair_hold, "HOLDOUT")
show_book({label: book_vs_ranking(full, p, HOLDOUT_START) for label, p in cands.items()},
          "HOLDOUT")
print(f"composite fair IC by calendar year, {GATES['ic_horizon']} bars ahead "
      "(train years from the train panel, 2025+ from the full one):")
display(pd.DataFrame({label: pd.concat([fair_train[label][1].loc["COMPOSITE"],
                                        fair_hold[label][1].loc["COMPOSITE"]])
                      for label in cands}).T.style.format("{:+.3f}"))

### 5c · The holdout book, name by name — and which rule traded it (FOCUS)

The equity curve, then who was held, when each name was bought and sold, how much of the book it
occupied and what it contributed (`plots.py` beside `strategy.py`), then **why each hold
ended**. That last table has caught this strategy's two worst bugs — displacement owning 122 of
127 exits (v16) and the `min_events` eligibility cliff owning 140 of 196 (v20) — both invisible in
an equity curve. If one mechanism owns most exits, that mechanism *is* the strategy; a high
`score_held` means the name was not sold because its thesis decayed.

In [ ]:
import importlib.util

pf_hold = hold[FOCUS]["pf"]
plots.plot_equity(bt.equity_curve(pf_hold), title=f"Holdout equity — {FOCUS}").show()
_spec = importlib.util.spec_from_file_location("informed_flow_plots", strat.dir / "plots.py")
if_plots = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(if_plots)
if_plots.plot_holdings(holdout, pf_hold).show()

causes = mod["exit_causes"](full, **PARAMS)                 # warm, like the holdout book
causes = causes[causes["exit"] >= pd.Timestamp(HOLDOUT_START, tz="UTC")].reset_index(drop=True)
n = len(causes)
print(f"HOLDOUT ({FOCUS}): {n} completed holds "
      f"({int((bt.trade_records(pf_hold)['status'] == 'Open').sum())} still open)")
if n:
    display(causes.groupby("cause")
            .agg(exits=("cause", "size"), share=("cause", lambda s: len(s) / n),
                 median_bars_held=("bars_held", "median"),
                 median_score_held=("score_held", "median"),
                 still_above_q_in=("score_held", lambda s: (s >= PARAMS["q_in"]).mean()))
            .sort_values("exits", ascending=False)
            .style.format({"share": "{:.0%}", "median_score_held": "{:.3f}",
                           "still_above_q_in": "{:.0%}"}))
    short = causes[causes["bars_held"] <= 5]
    print(f"holds of <= 5 bars: {len(short)}/{n}")
    display(causes.sort_values("bars_held").head(8))

### 5d · Shuffled books on the holdout — the signal on unseen data

§3e's test, on the window the study never saw: the candidate's own book — same slots, exit band,
costs and fills — driven by the real ranking and by `N_OOS_SIMS` symbol-shuffled ones, all scored
warm from the holdout start. Because the shuffles pay the same costs and run the same rules,
**real minus shuffled mean** is the cleanest number for selection skill there is: construction
drag, turnover and the luck of holding 30 names cancel out of it. And because it is *relative*,
it is far less worn by earlier reads of this window than the absolute numbers in §5a.

It runs twice. **Shuffled freely**: any name can receive any other name's score history.
**Shuffled within dollar-volume tenths** (fixed from the universe's own 2020-H1 ranking window, the
same tenths as §4's walk-forward shuffle): short interest leans toward heavily traded names (rank correlation ≈ 0.4
with dollar volume), and those names beat an *equal-weight* benchmark through 2023–26 whatever
their short interest said. A free shuffle would credit that lean as skill; inside a tenth (50
names of similar liquidity) only the choice *among alike names* is left. A real edge beats
**both** 95th percentiles.

In [ ]:
rng_oos = np.random.default_rng(11)             # its own stream: §3e's numbers must not move
KINDS = {"free": None, "within tenths": TENTHS}
oos_shuffle = {}
for label, params in cands.items():
    p = {**strat.params, **params}
    parts = compose(full, **p)
    real = book_hurdle(parts, p, parts["score_pct"], full, HOLDOUT_START)
    if abs(real - hold[label][HURDLE]) > 1e-6:
        print(f"⚠ {label}: rebuilt book {real:+.4f} ≠ §5a {hold[label][HURDLE]:+.4f} — the "
              "shuffle is not running the strategy's own machinery")
    row = {"real ranking": real}
    for kind, groups in KINDS.items():
        sims = np.array([book_hurdle(parts, p, shuffled(parts, rng_oos, groups), full,
                                     HOLDOUT_START) for _ in range(N_OOS_SIMS)])
        row |= {f"{kind}: mean": sims.mean(), f"{kind}: p{PCTL}": np.percentile(sims, PCTL),
                f"{kind}: real − mean": real - sims.mean(),
                f"{kind}: shuffles ≥ real": float((sims >= real).mean())}
    oos_shuffle[label] = row
oos_shuffle = pd.DataFrame(oos_shuffle).T
display(oos_shuffle.style.format(PCT)
        .format("{:.0%}", subset=[c for c in oos_shuffle.columns if c.endswith("≥ real")]))
print(f"holdout {HOLDOUT_START} → {LAST}, {HURDLE}; {N_OOS_SIMS} shuffles per kind")

### 5e · Can a real order get the open? (blocker 2)

Every number above fills at the **stored daily open**. A live book can only earn it with
**market-on-open orders** — the opening auction — so this checks the premise against the
1-minute table, for every name of the panel it covers: how far the stored open sits from the
**first regular-session trade** (what the auction prints), and how far the **first half hour's
volume-weighted price** sits from it (what an order placed by hand after the open gets). The
last row asks the same of the FOCUS book's **own holdout orders**, signed so that a positive
number is a cost: does this book buy names that run up after the open (or sell ones that fall)?
The 1-minute history starts 2025-06-18 and covers only part of the universe, so read `n`.

In [ ]:
with engine.connect() as c:
    opens = pd.read_sql(text("""
        WITH m AS (
          SELECT symbol, (ts AT TIME ZONE 'America/New_York')::date AS d,
                 (array_agg(open ORDER BY ts))[1] AS first_trade,
                 sum((high + low + close) / 3 * volume) / nullif(sum(volume), 0) AS vwap30
          FROM prices_ohlcv
          WHERE bar = '1m' AND rth AND symbol = ANY(:s)
            AND (ts AT TIME ZONE 'America/New_York')::time < '10:00'
          GROUP BY 1, 2)
        SELECT m.*, x.open AS daily_open FROM m JOIN prices_ohlcv x
          ON x.symbol = m.symbol AND x.bar = '1d' AND (x.ts AT TIME ZONE 'UTC')::date = m.d"""),
        c, params={"s": list(full.symbols)}, parse_dates=["d"])
opens["first trade"] = (opens["first_trade"] / opens["daily_open"] - 1.0) * 1e4
opens["first half hour"] = (opens["vwap30"] / opens["daily_open"] - 1.0) * 1e4

orders = hold[FOCUS]["pf"].orders.records_readable
book = pd.DataFrame({
    "symbol": orders["Column"],
    "d": pd.to_datetime(orders["Timestamp"]).dt.tz_convert("UTC").dt.tz_localize(None)
         .dt.normalize(),
    "side": np.where(orders["Side"] == "Buy", 1.0, -1.0)}).merge(opens, on=["symbol", "d"])
book_cost = book["side"] * book["first half hour"]           # + = a late fill costs this book


def spread(x):
    """n, median and 90th percentile of the absolute distance, and the signed mean, in bp."""
    return {"n": len(x), "median |bp|": x.abs().median(), "p90 |bp|": x.abs().quantile(0.9),
            "mean bp": x.mean()}


OPEN_CHECK = pd.DataFrame({
    "stored open vs the first trade": spread(opens["first trade"]),
    "first-half-hour VWAP vs the open": spread(opens["first half hour"]),
    f"{FOCUS}'s own holdout orders, first half hour (signed cost)": spread(book_cost),
}).T
display(OPEN_CHECK.style.format("{:.1f}").format("{:,.0f}", subset=["n"]))
print(f"{opens['symbol'].nunique()} of {len(full.symbols)} names have 1-minute bars "
      f"({opens['d'].min().date()} → {opens['d'].max().date()}); a market-on-open order earns the "
      "stored open, and the gate in §3a charges a late fill "
      f"{GATES['late_fill_bps']:g} bp on EVERY leg, adversely")

## 6 · Verdict

One row per gate, one column per candidate, in the order of the three questions: **1** is there
skill, **2** is it worth the risk, **3** can it be run. A **FAIL** on any hard gate fails the
candidate; **WARN** is a soft gate (everything measured on the used holdout alone, your own risk
limits, the fill-timing stress — and since v32 the holdout shuffle too) and **INFO** is reported without a threshold — all set in §0
`GATES`. The walk-forward rows judge the optimization procedure, so they read the same for every
candidate.

In [ ]:
def gate(ok, value, hard=True):
    """One scoreboard cell: PASS / FAIL (WARN for a soft gate) and the number behind it."""
    return f"{'PASS' if ok else 'FAIL' if hard else 'WARN'} · {value}"


g = GATES
wf_skill = (WF_STITCHED["alpha"] > 0 and WF_STITCHED["P(alpha > 0)"] >= g["min_prob_alpha"]
            and g["wf_min_positive_share"] * len(oos_x) <= WF_POSITIVE)
wf_worth = (WF_STITCHED["cagr"] > WF_STITCHED["buy & hold"]
            and WF_STITCHED["return / vol"] >= WF_STITCHED["B&H return / vol"])
naive, fixed = WF_ARMS.loc["flip · all five sleeves, equal weight"], WF_ARMS.loc[FIXED_ARM]
popular = WF_ARMS.loc[NAIVE_ARM]
TWICE = f"{g['cost_multiple']:g}x costs"
board = {}
for label in cands:
    tr, h, shuf = train[label], hold[label], oos_shuffle.loc[label]
    rule_ok, rule_text = signal_rule(fair_train[label][0].loc["COMPOSITE"])
    ic_hold = fair_hold[label][0].loc["COMPOSITE"]
    fills = {f: costs.loc[label, [c for c in costs.columns if c.startswith(f"fill: {f}")][0]]
             for f in ("open", "pessimistic")}
    worst_year = min(h["year_alpha"].values()) if h["year_alpha"] else np.nan
    cells = {
        # 1 · is there skill? --------------------------------------------------------------------
        "1 · signal rule · train": gate(rule_ok, rule_text),
        "1 · walk-forward alpha (procedure)": gate(
            wf_skill, f"alpha {WF_STITCHED['alpha']:+.1%} · P(alpha > 0) "
                      f"{WF_STITCHED['P(alpha > 0)']:.0%} · {WF_POSITIVE}/{len(oos_x)} periods"),
        f"1 · rule arm beats shuffled p{PCTL} · whole walk-forward": gate(
            WF_SHUFFLE["real"] > WF_SHUFFLE[f"shuffled p{PCTL}"],
            f"real {WF_SHUFFLE['real']:+.1%} vs p{PCTL} {WF_SHUFFLE[f'shuffled p{PCTL}']:+.1%} "
            f"(inside tenths) · real − mean {WF_SHUFFLE['real − mean']:+.1%} · "
            f"{WF_SHUFFLE['shuffles ≥ real']:.0%} of shuffles ≥ real"),
        # 2 · is it worth the risk? --------------------------------------------------------------
        "2 · walk-forward beats buy & hold": gate(
            wf_worth, f"CAGR {WF_STITCHED['cagr']:+.1%} vs {WF_STITCHED['buy & hold']:+.1%} · "
                      f"return/vol {WF_STITCHED['return / vol']:.2f} vs "
                      f"{WF_STITCHED['B&H return / vol']:.2f}"),
        # 3 · can it be run? ---------------------------------------------------------------------
        "3 · completed holds · train": gate(
            tr["holds_per_year"] >= g["min_holds_per_year"],
            f"{tr['holds_per_year']:.0f}/yr vs {g['min_holds_per_year']} "
            f"({tr['trades_per_year']:.0f} lots/yr)"),
        "3 · alpha at the real order size · train": gate(
            costs.loc[label, REAL_SIZE] > 0, f"{costs.loc[label, REAL_SIZE]:+.1%}"),
        f"3 · alpha at {TWICE} · train": gate(costs.loc[label, TWICE] > 0,
                                              f"{costs.loc[label, TWICE]:+.1%}"),
        "3 · no stability spike": gate(stability[label]["spikes"] == 0,
                                       f"{stability[label]['spikes']} spike(s)"),
        f"3 · alpha with {LATE_COL} · train": gate(costs.loc[label, LATE_COL] > 0,
                                                    f"{costs.loc[label, LATE_COL]:+.1%}"),
        f"3 · alpha with {WHOLE_COL} · train": gate(costs.loc[label, WHOLE_COL] > 0,
                                                     f"{costs.loc[label, WHOLE_COL]:+.1%}"),
        # warnings: the used holdout alone (read under eleven versions), your risk limits ---------
        f"beats shuffled p{PCTL} · holdout": gate(
            all(shuf["real ranking"] > shuf[f"{k}: p{PCTL}"] for k in KINDS),
            f"real {shuf['real ranking']:+.1%} vs p{PCTL} "
            + " / ".join(f"{shuf[f'{k}: p{PCTL}']:+.1%} ({k})" for k in KINDS)
            + f" · real − mean {shuf['free: real − mean']:+.1%}", hard=False),
        "alpha with real execution · holdout": gate(
            min(h["alpha_late"], h["alpha_whole"]) > 0,
            f"{LATE_COL} {h['alpha_late']:+.1%} · {WHOLE_COL} {h['alpha_whole']:+.1%}",
            hard=False),
        "alpha after beta · holdout": gate(h["alpha_cagr"] > 0,
                                           f"{h['alpha_cagr']:+.1%} (beta {h['beta']:.2f})",
                                           hard=False),
        "P(alpha > 0) · holdout bootstrap": gate(
            h["P(alpha > 0)"] >= g["min_prob_alpha"],
            f"{h['P(alpha > 0)']:.0%} · 90% CI {h['alpha p5']:+.1%} … {h['alpha p95']:+.1%}",
            hard=False),
        "alpha > 0 every holdout year": gate(worst_year > 0, f"worst {worst_year:+.1%}",
                                             hard=False),
        "beats buy & hold · holdout": gate(
            h["excess_cagr"] > 0 and h["ret_per_vol"] >= h["bench_ret_per_vol"],
            f"excess {h['excess_cagr']:+.1%} · return/vol {h['ret_per_vol']:.2f} vs "
            f"{h['bench_ret_per_vol']:.2f}", hard=False),
        "ranking selects · holdout fair IC": gate(
            ic_hold["IC"] > 0 and ic_hold["t"] >= 1.0,
            f"{ic_hold['IC']:+.3f} (t {ic_hold['t']:+.1f}) at {g['ic_horizon']} bars", hard=False),
        "beta · holdout": gate(h["beta"] <= g["max_beta"],
                               f"{h['beta']:.2f} (limit {g['max_beta']})", hard=False),
        "max drawdown · holdout": gate(
            h["max_drawdown"] >= g["max_drawdown"],
            f"{h['max_drawdown']:+.1%} (limit {g['max_drawdown']:+.0%})", hard=False),
        "volatility · holdout": gate(
            h["volatility"] <= g["max_vol_ratio"] * h["bench_volatility"],
            f"{h['volatility']:.1%} vs {g['max_vol_ratio']}× {h['bench_volatility']:.1%}",
            hard=False),
        "fill timing · train alpha": gate(
            fills["pessimistic"] >= g["stress_min_alpha"],
            f"open {fills['open']:+.1%} → pessimistic {fills['pessimistic']:+.1%} "
            f"(limit {g['stress_min_alpha']:+.0%})", hard=False),
        "choosing sleeves earns its place · walk-forward": gate(
            WF_STITCHED["alpha"] >= naive["alpha"],
            f"rule arm {WF_STITCHED['alpha']:+.1%} vs all five sleeves {naive['alpha']:+.1%} · "
            f"config sleeves fixed {fixed['alpha']:+.1%} (P {fixed['P(alpha > 0)']:.0%})",
            hard=False),
        "sleeves the rule chose · walk-forward": f"INFO · {SLEEVE_VOTES}",
        "the optimizer earns its place (§4d, paired)": (
            f"INFO · re-tuned − untuned rule arm Δ {VS_UNTUNED['Δ alpha']:+.1%} "
            f"(P {VS_UNTUNED['P(Δ > 0)']:.0%}, {VS_UNTUNED['periods won']} periods)"
            if VS_UNTUNED else "INFO · nothing is searched"),
        "against the adopted version (§4d, paired)": (
            "INFO · " + " · ".join(
                f"{k}: Δ {v['Δ alpha']:+.1%} (P {v['P(Δ > 0)']:.0%}, "
                f"{v.get('periods won', '?')} periods, train Δ {v['Δ train alpha']:+.1%})"
                for k, v in VS_BASELINE.items())
            + (f" · leave-top-5-out Δ active {LEAVE5['Δ active, without them']:+.1%}"
               if LEAVE5 else "") if VS_BASELINE else "INFO · no baseline"),
        "the open is fillable (§5e, 1-minute data)": (
            "INFO · first trade vs stored open: median "
            f"{OPEN_CHECK.iloc[0]['median |bp|']:.1f} bp "
            f"(p90 {OPEN_CHECK.iloc[0]['p90 |bp|']:.1f}) · "
            f"first half hour: median {OPEN_CHECK.iloc[1]['median |bp|']:.0f} bp · this book's "
            f"late-fill cost {OPEN_CHECK.iloc[2]['mean bp']:+.1f} bp/order "
            f"(n {OPEN_CHECK.iloc[2]['n']:.0f})"),
        "beats the 50 most traded names · walk-forward": gate(
            WF_STITCHED["cagr"] > popular["cagr"],
            f"rule arm {WF_STITCHED['cagr']:+.1%}/yr vs naive basket {popular['cagr']:+.1%}/yr "
            f"(return/vol {WF_STITCHED['return / vol']:.2f} vs {popular['return / vol']:.2f})",
            hard=False),
        "SPY, same windows": (
            f"INFO · walk-forward {window_cagr(SPY, first.date(), last.date()):+.1%}/yr · "
            f"holdout {window_cagr(SPY, HOLDOUT_START):+.1%}/yr"),
        "shuffled books · train": f"INFO · z {randoms.loc[label, 'z']:+.1f}, construction alone "
                                  f"{randoms.loc[label, 'shuffled mean']:+.1%}",
    }
    failed = [k for k, v in cells.items() if v.startswith("FAIL")]
    cells["VERDICT"] = "PASS" if not failed else f"FAIL ({len(failed)} gate(s))"
    board[label] = cells

scoreboard = pd.DataFrame(board)
COLOR = {"PASS": "#d8efd6", "FAIL": "#f6d5d1", "WARN": "#fbe8c6", "INFO": ""}
print(f"{STUDY} · train scored {SCORE_FROM} → {d['end']} · walk-forward {first.date()} → "
      f"{last.date()} · holdout {HOLDOUT_START} → {LAST} (used)")
display(scoreboard.style.map(lambda v: f"background: {COLOR.get(v.split(' ')[0], '')}"))
# One line a later run (or a person) can diff: the same numbers, every version.
import json


def rounded(obj):
    """Floats to 4 decimals, all the way down, so the scorecard stays one readable line."""
    if isinstance(obj, dict):
        return {k: rounded(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [rounded(v) for v in obj]
    return round(float(obj), 4) if isinstance(obj, (float, np.floating, np.integer)) else obj


tr, h = train[FOCUS], hold[FOCUS]
fair, fifths = fair_train[FOCUS][0].loc["COMPOSITE"], fair_train[FOCUS][2]
top_col = [c for c in fifths.columns if c.startswith("top ")][0]
pess_col = [c for c in costs.columns if c.startswith("fill: pessimistic")][0]
WF_KEYS = ("cagr", "buy & hold", "alpha", "P(alpha > 0)", "return / vol")
print("SCORECARD " + json.dumps(rounded({
    "study": STUDY, "universe": d["universe"], "symbols": len(syms),
    "rule": {"min_t": g["ic_min_t"], "year_test": g["year_test"]},
    "sleeves": [k for k, w in WEIGHT_OF.items() if strat.params[w] > 0],
    "verdict": scoreboard.loc["VERDICT", FOCUS],
    "train": {"alpha": tr["alpha_cagr"], "cagr": tr["cagr"], "bh": tr["bench_cagr"],
              "ret_vol": tr["ret_per_vol"], "bh_ret_vol": tr["bench_ret_per_vol"],
              "max_dd": tr["max_drawdown"], "holds_yr": tr["holds_per_year"],
              "fair_ic": fair["IC"], "fair_t": fair["t"],
              "top_k_fwd": fifths.loc["COMPOSITE", top_col],
              "alpha_2x": costs.loc[FOCUS, TWICE], "alpha_pessimistic": costs.loc[FOCUS, pess_col],
              "breakeven_bps": costs.loc[FOCUS, "breakeven round-trip bps"],
              "spikes": stability[FOCUS]["spikes"], "shuffle_z": randoms.loc[FOCUS, "z"],
              "alpha_late_fill": costs.loc[FOCUS, LATE_COL],
              "alpha_whole_shares": costs.loc[FOCUS, WHOLE_COL]},
    "wf": {"arms": {arm: {k: WF_ARMS.loc[arm, k] for k in WF_KEYS} for arm in WF_ARMS.index},
           "periods_positive": f"{WF_POSITIVE}/{len(oos_x)}", "votes": SLEEVE_VOTES,
           "shuffle": WF_SHUFFLE, "spy": window_cagr(SPY, first.date(), last.date())},
    "holdout": {"cagr": h["cagr"], "bh": h["bench_cagr"], "alpha": h["alpha_cagr"],
                "p_alpha": h["P(alpha > 0)"], "beta": h["beta"], "ret_vol": h["ret_per_vol"],
                "bh_ret_vol": h["bench_ret_per_vol"], "max_dd": h["max_drawdown"],
                "fair_ic": fair_hold[FOCUS][0].loc["COMPOSITE", "IC"],
                "spy": window_cagr(SPY, HOLDOUT_START),
                "alpha_late_fill": h["alpha_late"], "alpha_whole_shares": h["alpha_whole"],
                "shuffle": oos_shuffle.loc[FOCUS].to_dict()},
    "open_check": OPEN_CHECK.to_dict("index"),
    "vs_baseline": {"label": BASELINE_LABEL, **VS_BASELINE, "leave_top5": LEAVE5},
    "vs_untuned": VS_UNTUNED,
})))
print(f"the rule's latest reading (training data to {FOLDS[-1][0]}): "
      f"{wf[-1]['sleeves the rule chose']} — config.yaml ships "
      f"{' + '.join(k for k, w in WEIGHT_OF.items() if strat.params[w] > 0)}")

## 7 · Promote a candidate into `config.yaml`

Off by default, and deliberately so: this **rewrites a tracked file**, and what ships is a
decision, not a side effect of running a notebook. It refuses a candidate whose verdict failed
unless `FORCE = True` (say why in the commit). The rewrite is surgical — only the `value:` lines
change, every comment survives — plus one `# promoted:` provenance line above `params:` (study,
trial, date, holdout end, verdict), so the shipped values can always be traced back to the run
that justified them. It verifies itself by reloading the strategy, restoring the original file
if the config no longer loads or a value does not round-trip.

In [ ]:
import pathlib
import re

PROMOTE = FOCUS        # which candidate to write into config.yaml
WRITE_BACK = False     # True rewrites strategies/informed_flow/config.yaml
FORCE = False          # True writes a candidate that failed the verdict


def _yaml_scalar(v):
    """Render one value the way config.yaml writes it."""
    if isinstance(v, bool):
        return "true" if v else "false"
    return f'"{v}"' if isinstance(v, str) else repr(v)


def promote(params, *, provenance, apply=False):
    """Rewrite `params.<key>.value` for every knob that differs, stamp provenance; verify."""
    path = pathlib.Path(strat.dir) / "config.yaml"
    original = path.read_text()
    cfg = yaml.safe_load(original)
    lines, changes = original.splitlines(), []
    for key, new in params.items():
        spec = cfg["params"].get(key)
        if not isinstance(spec, dict) or spec.get("value") == new:
            continue
        at = next((i for i, ln in enumerate(lines) if re.fullmatch(rf"  {key}:\s*", ln)), None)
        if at is None:
            print(f"  ! {key}: no long-form entry to write to — edit it by hand")
            continue
        end = next((i for i in range(at + 1, len(lines))
                    if lines[i][:3].strip() and not lines[i].startswith("    ")), len(lines))
        for i in range(at + 1, end):
            if lines[i].split("#")[0].strip().startswith("value:"):
                note = lines[i].partition("#")[2]
                lines[i] = f"    value: {_yaml_scalar(new)}" + (f"   #{note}" if note else "")
        changes.append((key, spec.get("value"), new))
    for key, old, new in changes:
        print(f"  {key:18s} {old!r:>12} -> {new!r}")
    if not changes:
        print("config.yaml already ships these values — nothing to promote.")
        return changes
    print(f"  provenance: {provenance}")
    if not apply:
        print(f"\n{len(changes)} knob(s) would change. Set WRITE_BACK = True to apply.")
        return changes
    lines = [ln for ln in lines if not ln.startswith("# promoted:")]
    at = lines.index("params:")
    lines.insert(at, f"# promoted: {provenance}")
    path.write_text("\n".join(lines) + "\n")
    try:
        reloaded = strategies.load(strat.name)            # the config invariants run here
    except Exception:
        path.write_text(original)
        print("REVERTED — the rewritten config does not load:")
        raise
    if bad := {k: v for k, v in params.items() if reloaded.params.get(k) != v}:
        path.write_text(original)
        raise AssertionError(f"REVERTED — values did not round-trip: {bad}")
    print(f"\nwrote {path} — commit the diff, then re-run both notebooks from the top.")
    return changes


verdict = scoreboard.loc["VERDICT", PROMOTE]
print(f"{PROMOTE}: verdict {verdict}\n")
if WRITE_BACK and verdict != "PASS" and not FORCE:
    raise RuntimeError(f"{PROMOTE} failed the verdict — set FORCE = True to write it anyway")
PROVENANCE = (f"{STUDY} trial #{TRIAL_OF[PROMOTE]} · {pd.Timestamp.now(tz='UTC').date()} · "
              f"holdout {HOLDOUT_START}..{LAST} · verdict {verdict}"
              + (" · FORCED" if FORCE and verdict != "PASS" else ""))
_ = promote(cands[PROMOTE], provenance=PROVENANCE, apply=WRITE_BACK)

## Read the result honestly

- The **walk-forward** and the **shuffled books on the holdout** are the numbers that count; §3 can
  only show fragility, and the holdout's absolute numbers are warnings because that window has been
  read many times.
- The walk-forward judges the **procedure**: re-read the signal rule on the data so far, hold the
  sleeves that pass, trade the fixed rules. Its **rule arm** is the only arm that never saw its
  own test period in any form; the fixed arms are controls, and picking the best-looking one
  afterwards would be selection on the walk-forward.
- **Alpha, and the market too.** A long-only book that beats buy-and-hold only by holding
  higher-beta names has bought exposure (v25); one that earns alpha while trailing the index is
  not worth running (v26). A candidate needs both, and return ÷ volatility at least the market's.
- **The signal first.** The ranking makes ~470 small predictions a day; the book makes 30 large
  bets. §2a's fair check is where a small edge can actually be seen; a 30-name book needs a
  measured alpha of roughly +4–5%/yr over the whole stitched record to clear "90% sure", so a true
  edge of that size fails about half the time. No gate design fixes a small sample.
- **Fill timing decides everything.** At 3–4 book turnovers a year pessimistic fills cost more than
  the whole signal. The numbers here assume market-on-open orders fill at the open; if yours would
  not, believe the pessimistic column instead.
- Not modelled: taxes, and the fact that the 500-name equal-weight benchmark cannot be bought with
  this account — the real alternative to this strategy is an index fund.
- The universe is point-in-time from 2020-07 but holds only names still listed today, and the
  yfinance ratings sleeve is a current snapshot, not an archive; short interest (FINRA) is genuinely
  point-in-time. Keep claims to this liquid universe.
- A PASS is a reason to **paper-trade**, not to fund: the only data nobody has looked at is the
  future (IDEAS.md, "Production, only after a PASS").